In [1]:
#lets get it

In [2]:
import os
import datetime
from datetime import datetime
import re
import json
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Image processing
import torchvision.transforms as transforms
from torchvision.transforms import functional as TF
import random

# For stratified split
from sklearn.model_selection import train_test_split

# For parallel image downloads
import concurrent.futures
import urllib.request
from pathlib import Path
import time
from tqdm import tqdm

# Import CLIP - handle both installation methods
try:
    import clip
    CLIP_AVAILABLE = True
    print("CLIP loaded successfully")
except ImportError:
    CLIP_AVAILABLE = False
    print("CLIP not found. Installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/openai/CLIP.git"])
    import clip
    CLIP_AVAILABLE = True
    print("CLIP installed and loaded")

# Alternative: Use Hugging Face transformers if CLIP install fails
USE_HF_CLIP = False  # Set to True if standard CLIP fails
if USE_HF_CLIP or not CLIP_AVAILABLE:
    from transformers import CLIPModel, CLIPProcessor
    print("Using Hugging Face CLIP implementation")

# Hyperparameter Configuration - Modify these as needed
CONFIG = {
    # Data paths
    'train_csv': 'dataset/train.csv',
    'test_csv': 'dataset/test.csv',
    'image_dir': 'images/',
    'output_dir': 'outputs/',
    
    # Model settings
    'clip_model': 'ViT-B/32',  # Options: 'ViT-B/32', 'ViT-L/14'
    'image_size': 224,  # 224 for ViT-B/32, can use 336 for ViT-L/14
    'text_max_length': 77,  # CLIP's default
    'freeze_clip': True,  # Set False to finetune last layers
    'freeze_layers_until': -2,  # If freeze_clip=False, finetune last 2 layers
    
    # Training settings
    'batch_size': 64,  # Adjust based on GPU memory
    # 'learning_rate': 1e-4,
    'learning_rate': 3e-4,
    'epochs': 500,
    'early_stopping_patience': 100,
    'val_split': 0.2,
    'stratify_bins': 10,  # Number of price bins for stratified split
    
    # Loss weights
    'smape_weight': 0.5,
    'log_mse_weight': 0.3,
    'huber_weight': 0.2,
    
    # Augmentation settings
    'use_heavy_augmentations': True,
    'mixup_alpha': 0.2,
    'cutmix_alpha': 1.0,
    'augmentation_prob': 0.5,
    
    # Other settings
    'num_workers': 4,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42
}

# Set random seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

print(f"Using device: {CONFIG['device']}")
print(f"CLIP model: {CONFIG['clip_model']}")
print(f"Batch size: {CONFIG['batch_size']}")

# Test CLIP import
if CLIP_AVAILABLE and not USE_HF_CLIP:
    print("\nTesting CLIP...")
    available_models = clip.available_models()
    print(f"Available CLIP models: {available_models[:5]}...")  # Show first 5
    
    # Load model to test
    test_model, test_preprocess = clip.load(CONFIG['clip_model'], device='cpu')
    print(f"CLIP model loaded successfully: {CONFIG['clip_model']}")
    del test_model, test_preprocess  # Free memory

CLIP loaded successfully
Using device: cuda
CLIP model: ViT-B/32
Batch size: 64

Testing CLIP...
Available CLIP models: ['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64']...
CLIP model loaded successfully: ViT-B/32


In [3]:
# # Cell 1.5: CLIP Wrapper to Handle Both Implementations

# class CLIPWrapper:
#     """Wrapper to handle both OpenAI CLIP and HuggingFace CLIP implementations"""
    
#     def __init__(self, model_name='ViT-B/32', device='cuda', use_hf=False):
#         self.device = device
#         self.use_hf = use_hf
#         self.model_name = model_name
        
#         if not use_hf:
#             # Try OpenAI CLIP first
#             try:
#                 import clip
#                 if model_name == 'ViT-B/32':
#                     self.model, self.preprocess = clip.load('ViT-B/32', device=device)
#                 elif model_name == 'ViT-L/14':
#                     self.model, self.preprocess = clip.load('ViT-L/14', device=device)
#                 else:
#                     self.model, self.preprocess = clip.load(model_name, device=device)
                
#                 self.tokenizer = clip.tokenize
#                 self.text_dim = 512 if 'ViT-B' in model_name else 768
#                 self.vision_dim = 512 if 'ViT-B' in model_name else 768
#                 print(f"Loaded OpenAI CLIP: {model_name}")
                
#             except Exception as e:
#                 print(f"OpenAI CLIP failed: {e}")
#                 print("Falling back to HuggingFace CLIP...")
#                 self.use_hf = True
        
#         if self.use_hf:
#             # Use HuggingFace implementation
#             from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
            
#             if 'ViT-B' in model_name:
#                 hf_model_name = "openai/clip-vit-base-patch32"
#                 self.text_dim = 512
#                 self.vision_dim = 768
#             elif 'ViT-L' in model_name:
#                 hf_model_name = "openai/clip-vit-large-patch14"
#                 self.text_dim = 768
#                 self.vision_dim = 1024
#             else:
#                 hf_model_name = "openai/clip-vit-base-patch32"
#                 self.text_dim = 512
#                 self.vision_dim = 768
            
#             self.model = CLIPModel.from_pretrained(hf_model_name).to(device)
#             self.processor = CLIPProcessor.from_pretrained(hf_model_name)
#             self.tokenizer = CLIPTokenizer.from_pretrained(hf_model_name)
#             print(f"Loaded HuggingFace CLIP: {hf_model_name}")
    
#     def encode_text(self, text_or_tokens):
#         """Encode text to features"""
#         if self.use_hf:
#             if isinstance(text_or_tokens, list):
#                 # text_or_tokens is list of strings
#                 inputs = self.tokenizer(text_or_tokens, padding=True, truncation=True, 
#                                        max_length=77, return_tensors="pt").to(self.device)
#                 outputs = self.model.get_text_features(**inputs)
#             else:
#                 # text_or_tokens is already tokenized tensor
#                 outputs = self.model.get_text_features(input_ids=text_or_tokens)
#             return F.normalize(outputs, dim=-1)
#         else:
#             # OpenAI CLIP
#             if isinstance(text_or_tokens, list):
#                 tokens = clip.tokenize(text_or_tokens, truncate=True).to(self.device)
#                 features = self.model.encode_text(tokens)
#             else:
#                 features = self.model.encode_text(text_or_tokens)
#             return features / features.norm(dim=-1, keepdim=True)
    
#     def encode_image(self, images):
#         """Encode images to features"""
#         if self.use_hf:
#             outputs = self.model.get_image_features(pixel_values=images)
#             return F.normalize(outputs, dim=-1)
#         else:
#             features = self.model.encode_image(images)
#             return features / features.norm(dim=-1, keepdim=True)
    
#     def tokenize(self, texts, truncate=True):
#         """Tokenize text"""
#         if self.use_hf:
#             return self.tokenizer(texts, padding=True, truncation=truncate, 
#                                 max_length=77, return_tensors="pt")['input_ids']
#         else:
#             return clip.tokenize(texts, truncate=truncate)
    
#     def preprocess_image(self, image):
#         """Preprocess image for model"""
#         if self.use_hf:
#             return self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
#         else:
#             return self.preprocess(image)


# # Updated CLIPFeatureExtractor to use wrapper
# class CLIPFeatureExtractor(nn.Module):
#     """Extract features using CLIP encoders with wrapper"""
    
#     def __init__(self, clip_model_name='ViT-B/32', device='cuda', use_hf=False):
#         super().__init__()
#         self.device = device
        
#         # Use wrapper for compatibility
#         self.clip_wrapper = CLIPWrapper(clip_model_name, device, use_hf)
        
#         # Get feature dimensions
#         self.text_dim = self.clip_wrapper.text_dim
#         self.vision_dim = self.clip_wrapper.vision_dim
        
#         print(f"CLIP text dimension: {self.text_dim}")
#         print(f"CLIP vision dimension: {self.vision_dim}")
        
#         # Freeze CLIP if specified
#         if CONFIG['freeze_clip']:
#             for param in self.clip_wrapper.model.parameters():
#                 param.requires_grad = False
#             print("CLIP weights frozen")
#         else:
#             # Freeze all except last few layers
#             if not self.clip_wrapper.use_hf:
#                 for name, param in self.clip_wrapper.model.named_parameters():
#                     if 'ln_final' not in name and 'text_projection' not in name and 'visual.ln_post' not in name:
#                         param.requires_grad = False
#             else:
#                 # For HuggingFace model
#                 for name, param in self.clip_wrapper.model.named_parameters():
#                     if 'final' not in name and 'projection' not in name:
#                         param.requires_grad = False
#             print(f"CLIP partially frozen - finetuning final layers only")
    
#     def encode_text(self, text_tokens):
#         """Encode text using CLIP"""
#         return self.clip_wrapper.encode_text(text_tokens)
    
#     def encode_image(self, images):
#         """Encode images using CLIP"""
#         return self.clip_wrapper.encode_image(images)
    
#     def forward(self, text_tokens, images):
#         """Extract both text and image features"""
#         text_features = self.encode_text(text_tokens)
#         image_features = self.encode_image(images)
#         return text_features, image_features


# # Updated Dataset class to work with wrapper
# class ProductPriceDataset(Dataset):
#     """Multimodal dataset with heavy augmentations"""
    
#     def __init__(self, df, image_dir, clip_wrapper, text_preprocessor, is_train=True):
#         self.df = df
#         self.image_dir = image_dir
#         self.clip_wrapper = clip_wrapper
#         self.text_preprocessor = text_preprocessor
#         self.is_train = is_train
        
#         # Heavy augmentations for training
#         if is_train and CONFIG['use_heavy_augmentations']:
#             self.augmentation = transforms.Compose([
#                 transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.7, 1.0)),
#                 transforms.RandomHorizontalFlip(p=0.5),
#                 transforms.RandomRotation(degrees=15),
#                 transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
#                 transforms.RandomGrayscale(p=0.1),
#                 transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
#                 transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
#             ])
#             # Random erasing applied after normalization
#             self.random_erasing = transforms.RandomErasing(p=0.3, scale=(0.02, 0.2))
#         else:
#             self.augmentation = None
    
#     def __len__(self):
#         return len(self.df)
    
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
        
#         # Process text
#         cleaned_text, pack_count = self.text_preprocessor.prepare_for_clip(row['catalog_content'])
        
#         # Tokenize text for CLIP
#         text_tokens = self.clip_wrapper.tokenize([cleaned_text], truncate=True)[0]
        
#         # Load and process image
#         image_filename = row['image_link'].split('/')[-1]
#         image_path = os.path.join(self.image_dir, image_filename)
        
#         try:
#             image = Image.open(image_path).convert('RGB')
            
#             # Apply heavy augmentations during training
#             if self.is_train and self.augmentation and random.random() < CONFIG['augmentation_prob']:
#                 image = self.augmentation(image)
            
#             # Apply CLIP preprocessing
#             image = self.clip_wrapper.preprocess_image(image)
            
#             # Apply random erasing after normalization
#             if self.is_train and CONFIG['use_heavy_augmentations'] and random.random() < 0.3:
#                 image = self.random_erasing(image)
            
#         except Exception as e:
#             # Fallback to zero tensor if image loading fails
#             image = torch.zeros(3, CONFIG['image_size'], CONFIG['image_size'])
        
#         # Get price (log transform for stability)
#         price = np.log1p(row['price']) if 'price' in row else 0.0
        
#         return {
#             'text_tokens': text_tokens,
#             'image': image,
#             'pack_count': torch.tensor(pack_count, dtype=torch.float32),
#             'price': torch.tensor(price, dtype=torch.float32),
#             'price_original': torch.tensor(row.get('price', 0.0), dtype=torch.float32),
#             'sample_id': row['sample_id'],
#             'idx': idx
#         }


# # Quick test function
# def test_clip_setup():
#     """Test if CLIP is working correctly"""
#     print("\nTesting CLIP setup...")
    
#     # Try with OpenAI CLIP first
#     try:
#         wrapper = CLIPWrapper('ViT-B/32', 'cpu', use_hf=False)
        
#         # Test text encoding
#         test_text = ["a photo of a cat", "a photo of a dog"]
#         text_features = wrapper.encode_text(test_text)
#         print(f"Text features shape: {text_features.shape}")
        
#         # Test image encoding
#         test_image = torch.randn(1, 3, 224, 224)
#         image_features = wrapper.encode_image(test_image)
#         print(f"Image features shape: {image_features.shape}")
        
#         print("✓ CLIP setup successful!")
#         return True
        
#     except Exception as e:
#         print(f"Error: {e}")
#         print("Trying HuggingFace CLIP...")
        
#         try:
#             wrapper = CLIPWrapper('ViT-B/32', 'cpu', use_hf=True)
#             print("✓ HuggingFace CLIP setup successful!")
#             return True
#         except Exception as e2:
#             print(f"Both implementations failed: {e2}")
#             return False

# # Run test
# if test_clip_setup():
#     print("\nReady to proceed with training!")
# else:
#     print("\nPlease install CLIP manually:")
#     print("Option 1: pip install git+https://github.com/openai/CLIP.git")
#     print("Option 2: pip install transformers")

In [4]:
def download_single_image(args):
    """Download a single image with retry logic"""
    image_url, save_path, sample_id = args
    max_retries = 3
    
    if os.path.exists(save_path):
        return True, sample_id
    
    for attempt in range(max_retries):
        try:
            urllib.request.urlretrieve(image_url, save_path)
            return True, sample_id
        except Exception as e:
            if attempt == max_retries - 1:
                return False, sample_id
            time.sleep(1)  # Wait before retry
    
    return False, sample_id

def download_images_parallel(df, image_dir, max_workers=20):
    """Download images in parallel with progress bar"""
    os.makedirs(image_dir, exist_ok=True)
    failed_downloads = []
    
    # Prepare download tasks
    download_tasks = []
    for _, row in df.iterrows():
        image_url = row['image_link']
        filename = image_url.split('/')[-1]
        save_path = os.path.join(image_dir, filename)
        download_tasks.append((image_url, save_path, row['sample_id']))
    
    # Download in parallel
    print(f"Downloading {len(download_tasks)} images...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(
            executor.map(download_single_image, download_tasks),
            total=len(download_tasks),
            desc="Downloading images"
        ))
    
    # Log failed downloads
    for success, sample_id in results:
        if not success:
            failed_downloads.append(sample_id)
    
    # Save failed downloads to file
    if failed_downloads:
        with open('image_download_fail.txt', 'w') as f:
            for sample_id in failed_downloads:
                f.write(f"sample_id_{sample_id}.jpg failed to download\n")
        print(f"Failed to download {len(failed_downloads)} images. See image_download_fail.txt")
    else:
        print("All images downloaded successfully!")
    
    return failed_downloads

# Download images for training data
train_df = pd.read_csv(CONFIG['train_csv'])
failed_ids = download_images_parallel(train_df, CONFIG['image_dir'])


Failed to download 1 images. See image_download_fail.txt


In [5]:
class TextPreprocessor:
    """Clean and prepare text for CLIP encoding"""
    
    def __init__(self):
        # Common stopwords except units and important product words
        self.stopwords = set([
            'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for',
            'from', 'has', 'he', 'in', 'is', 'it', 'its', 'of', 'on',
            'that', 'the', 'to', 'was', 'will', 'with', 'this', 'these',
            'those', 'i', 'you', 'we', 'they', 'them', 'their', 'what',
            'which', 'who', 'when', 'where', 'why', 'how', 'than', 'then'
        ])
        
        # Important units to keep
        self.keep_units = set([
            'oz', 'ounce', 'pound', 'lb', 'gram', 'g', 'kg', 'kilogram',
            'liter', 'l', 'ml', 'milliliter', 'gallon', 'quart', 'pint',
            'cup', 'tablespoon', 'teaspoon', 'count', 'piece', 'pack',
            'fl', 'fluid', 'bottle', 'can', 'jar', 'box', 'bag'
        ])
    
    def extract_pack_count(self, text):
        """Extract pack count from text"""
        pack_match = re.search(r'\(Pack of (\d+)\)', text, re.IGNORECASE)
        if pack_match:
            return int(pack_match.group(1))
        
        # Check for other pack patterns
        pack_patterns = [
            r'(\d+)\s*pack',
            r'(\d+)\s*count',
            r'(\d+)\s*ct\b',
            r'(\d+)\s*pc\b',
            r'(\d+)\s*piece'
        ]
        
        for pattern in pack_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                pack_count = int(match.group(1))
                if 1 < pack_count < 100:  # Reasonable pack size
                    return pack_count
        
        return 1
    
    def extract_quantities(self, text):
        """Extract all numerical quantities from text"""
        # Pattern to match numbers with optional decimals
        pattern = r'\b\d+\.?\d*\b'
        quantities = re.findall(pattern, text)
        return [float(q) for q in quantities]
    
    def clean_text(self, text):
        """Clean and structure text for CLIP"""
        if pd.isna(text) or not text:
            return ""
        
        # Basic cleaning
        text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
        text = re.sub(r'[^\w\s\.\,\-\(\)\/\&]', '', text)  # Keep basic punctuation
        
        # Extract pack count for separate feature
        pack_count = self.extract_pack_count(text)
        
        # Split into words and filter
        words = text.lower().split()
        filtered_words = []
        
        for word in words:
            # Keep if it's a unit, number, or not a stopword
            if (word in self.keep_units or 
                re.match(r'\d+', word) or 
                word not in self.stopwords):
                filtered_words.append(word)
        
        # Reconstruct text
        cleaned = ' '.join(filtered_words)
        
        # Limit length for CLIP (it has 77 token limit)
        words = cleaned.split()
        if len(words) > 70:
            cleaned = ' '.join(words[:70])
        
        return cleaned, pack_count
    
    def prepare_for_clip(self, catalog_content):
        """Prepare text for CLIP encoding"""
        cleaned_text, pack_count = self.clean_text(catalog_content)
        
        # [TRY NEXT] - Extract brand using NER or pattern matching
        # brand = self.extract_brand_ner(cleaned_text)
        
        # [TRY NEXT] - Extract product category using zero-shot CLIP classification
        # category_probs = self.get_clip_category_probs(cleaned_text)
        
        return cleaned_text, pack_count
    
    
class NoCleanTextPreprocessor:
    def prepare_for_clip(self, catalog_content):
        if pd.isna(catalog_content):
            return ""
        # NO cleaning - just truncate if too long
        text = str(catalog_content)
        if len(text) > 500:  # CLIP can handle ~77 tokens = ~500 chars
            # Keep first 400 + last 100 chars to preserve title and key details
            text = text[:400] + " " + text[-100:]
        return text
    
    

# text_preprocessor = TextPreprocessor()
text_preprocessor = NoCleanTextPreprocessor()

In [6]:
# # Cell 3.5: Pre-process and Pre-tokenize Text Once

# def preprocess_dataframe(df, text_preprocessor):
#     """Pre-compute cleaned text and pack counts for entire dataframe"""
#     print("Pre-processing text and extracting pack counts...")
    
#     cleaned_texts = []
#     pack_counts = []
    
#     for catalog_content in tqdm(df['catalog_content'], desc="Preprocessing"):
#         cleaned, pack = text_preprocessor.prepare_for_clip(catalog_content)
#         cleaned_texts.append(cleaned)
#         pack_counts.append(pack)
    
#     df['cleaned_text'] = cleaned_texts
#     df['pack_count'] = pack_counts
    
#     print(f"Pack count distribution: {df['pack_count'].value_counts().head()}")
    
#     return df

# def batch_tokenize(texts, batch_size=1024):
#     """Tokenize texts in batches for efficiency"""
#     print(f"Tokenizing {len(texts)} texts...")
    
#     all_tokens = []
#     for i in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
#         batch_texts = texts[i:i+batch_size]
#         tokens = clip.tokenize(batch_texts, truncate=True)
#         all_tokens.append(tokens)
    
#     return torch.cat(all_tokens, dim=0)

# # Preprocess all dataframes
# train_df_full = preprocess_dataframe(train_df_full, text_preprocessor)
# train_df = preprocess_dataframe(train_df, text_preprocessor)
# val_df = preprocess_dataframe(val_df, text_preprocessor)

# # Pre-tokenize all texts
# train_tokens_full = batch_tokenize(train_df_full['cleaned_text'].tolist())
# train_tokens = batch_tokenize(train_df['cleaned_text'].tolist())
# val_tokens = batch_tokenize(val_df['cleaned_text'].tolist())

# print(f"Tokenized shapes - Train: {train_tokens.shape}, Val: {val_tokens.shape}")

# # Save tokenized data for later use
# torch.save({
#     'train_tokens': train_tokens,
#     'val_tokens': val_tokens,
#     'train_df': train_df,
#     'val_df': val_df
# }, 'outputs/preprocessed_data.pth')

# print("Saved preprocessed data to outputs/preprocessed_data.pth")

In [7]:
CONFIG['use_mild_augmentations'] = True
CONFIG['use_heavy_augmentations'] = False

class ProductPriceDataset(Dataset):
    """Multimodal dataset with heavy augmentations"""
    
    def __init__(self, df, image_dir, clip_model, clip_preprocess, 
                 text_preprocessor, is_train=True):
        self.df = df
        self.image_dir = image_dir
        self.clip_model = clip_model
        self.clip_preprocess = clip_preprocess
        self.text_preprocessor = text_preprocessor
        self.is_train = is_train
        
    #     # Heavy augmentations for training
    #     if is_train and CONFIG['use_heavy_augmentations']:
    #         self.augmentation = transforms.Compose([
    #             transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.7, 1.0)),
    #             transforms.RandomHorizontalFlip(p=0.5),
    #             transforms.RandomRotation(degrees=15),
    #             transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
    #             transforms.RandomGrayscale(p=0.1),
    #             transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    #             transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    #         ])
    #         # Random erasing applied after normalization
    #         self.random_erasing = transforms.RandomErasing(p=0.3, scale=(0.02, 0.2))
    #     else:
    #         self.augmentation = None
        
    #     # [TRY NEXT] - Add DINOv2 as alternative vision encoder
    #     # self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        
    # def __len__(self):
    #     return len(self.df)
    
    # def apply_cutmix(self, image1, image2, alpha=1.0):
    #     """Apply CutMix augmentation"""
    #     if alpha <= 0:
    #         return image1
        
    #     lam = np.random.beta(alpha, alpha)
        
    #     h, w = image1.shape[-2:]
    #     cut_rat = np.sqrt(1. - lam)
    #     cut_w = int(w * cut_rat)
    #     cut_h = int(h * cut_rat)
        
    #     cx = np.random.randint(w)
    #     cy = np.random.randint(h)
        
    #     bbx1 = np.clip(cx - cut_w // 2, 0, w)
    #     bby1 = np.clip(cy - cut_h // 2, 0, h)
    #     bbx2 = np.clip(cx + cut_w // 2, 0, w)
    #     bby2 = np.clip(cy + cut_h // 2, 0, h)
        
    #     mixed = image1.clone()
    #     mixed[:, bby1:bby2, bbx1:bbx2] = image2[:, bby1:bby2, bbx1:bbx2]
        
    #     return mixed
    
    # def __getitem__(self, idx):
    #     row = self.df.iloc[idx]
        
    #     # Process text
    #     cleaned_text, pack_count = self.text_preprocessor.prepare_for_clip(row['catalog_content'])
        
    #     # Tokenize text for CLIP
    #     text_tokens = clip.tokenize([cleaned_text], truncate=True)[0]
        
    #     # Load and process image
    #     image_filename = row['image_link'].split('/')[-1]
    #     image_path = os.path.join(self.image_dir, image_filename)
        
    #     try:
    #         image = Image.open(image_path).convert('RGB')
            
    #         # Apply heavy augmentations during training
    #         if self.is_train and self.augmentation and random.random() < CONFIG['augmentation_prob']:
    #             image = self.augmentation(image)
            
    #         # Apply CLIP preprocessing
    #         image = self.clip_preprocess(image)
            
    #         # Apply random erasing after normalization
    #         if self.is_train and CONFIG['use_heavy_augmentations'] and random.random() < 0.3:
    #             image = self.random_erasing(image)
            
    #     except:
    #         # Fallback to zero tensor if image loading fails
    #         image = torch.zeros(3, CONFIG['image_size'], CONFIG['image_size'])
    # Mild augmentations (recommended for product images)
        if is_train and CONFIG.get('use_mild_augmentations', True):
            self.augmentation = transforms.Compose([
                transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.9, 1.0)),  # Minimal cropping
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Very mild color changes
            ])
            self.random_erasing = None  # No erasing for mild
            
        # Heavy augmentations (keep as option but not recommended)
        elif is_train and CONFIG.get('use_heavy_augmentations', False):
            self.augmentation = transforms.Compose([
                transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2),
                transforms.RandomGrayscale(p=0.1),
                transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            ])
            self.random_erasing = transforms.RandomErasing(p=0.3, scale=(0.02, 0.2))
        else:
            self.augmentation = None
            self.random_erasing = None
    def __len__(self):
        return len(self.df)

    
    # def __getitem__(self, idx):
    #     row = self.df.iloc[idx]
        
    #     # Process text
    #     cleaned_text, pack_count = self.text_preprocessor.prepare_for_clip(row['catalog_content'])
        
    #     cleaned_text = self.text_preprocessor.prepare_for_clip(row['catalog_content'])

    #     # Tokenize text for CLIP
    #     text_tokens = clip.tokenize([cleaned_text], truncate=True)[0]
        
    #     # Load and process image
    #     image_filename = row['image_link'].split('/')[-1]
    #     image_path = os.path.join(self.image_dir, image_filename)
        
    #     try:
    #         image = Image.open(image_path).convert('RGB')
            
    #         # Apply augmentations during training
    #         if self.is_train and self.augmentation and random.random() < CONFIG.get('augmentation_prob', 0.5):
    #             image = self.augmentation(image)
            
    #         # Apply CLIP preprocessing
    #         image = self.clip_preprocess(image)
            
    #         # Apply random erasing if using heavy augmentations
    #         if self.is_train and self.random_erasing and random.random() < 0.3:
    #             image = self.random_erasing(image)
            
    #     except:
    #         image = torch.zeros(3, CONFIG['image_size'], CONFIG['image_size'])
        
    #     # Get price (log transform for stability)
    #     price = np.log1p(row['price']) if 'price' in row else 0.0
        
    #     # [TRY NEXT] - Add visual feature extraction beyond CLIP
    #     # vit_features = self.extract_vit_features(image)
    #     # color_histogram = self.extract_color_features(image)
        
    #     return {
    #         'text_tokens': text_tokens,
    #         'image': image,
    #         'pack_count': torch.tensor(pack_count, dtype=torch.float32),
    #         'price': torch.tensor(price, dtype=torch.float32),
    #         'price_original': torch.tensor(row.get('price', 0.0), dtype=torch.float32),
    #         'sample_id': row['sample_id'],
    #         'idx': idx  # For CutMix/MixUp
    #     }
    #no pack count please->
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # text
        cleaned_text = self.text_preprocessor.prepare_for_clip(row['catalog_content'])
        text_tokens = clip.tokenize([cleaned_text], truncate=True)[0]

        # image
        image_filename = row['image_link'].split('/')[-1]
        image_path = os.path.join(self.image_dir, image_filename)
        try:
            image = Image.open(image_path).convert('RGB')
            if self.is_train and self.augmentation and random.random() < CONFIG.get('augmentation_prob', 0.5):
                image = self.augmentation(image)
            image = self.clip_preprocess(image)
            if self.is_train and self.random_erasing and random.random() < 0.3:
                image = self.random_erasing(image)
        except:
            image = torch.zeros(3, CONFIG['image_size'], CONFIG['image_size'])

        # price in log space (for caching)
        price = np.log1p(row['price']) if 'price' in row else 0.0

        return {
            'text_tokens': text_tokens,
            'image': image,
            'price': torch.tensor(price, dtype=torch.float32),
            'price_original': torch.tensor(row.get('price', 0.0), dtype=torch.float32),
            'sample_id': row['sample_id'],
            'idx': idx
        }

    


In [8]:
class CLIPFeatureExtractor(nn.Module):
    """Extract features using CLIP encoders"""
    
    def __init__(self, clip_model_name='ViT-B/32', device='cuda'):
        super().__init__()
        self.device = device
        
        # Load CLIP model
        self.clip_model, self.preprocess = clip.load(clip_model_name, device=device)
        
        # Get feature dimensions
        self.text_dim = self.clip_model.encode_text(torch.zeros(1, 77).long().to(device)).shape[-1]
        self.vision_dim = self.clip_model.encode_image(torch.zeros(1, 3, 224, 224).to(device)).shape[-1]
        
        print(f"CLIP text dimension: {self.text_dim}")
        print(f"CLIP vision dimension: {self.vision_dim}")
        
        # Freeze CLIP if specified
        if CONFIG['freeze_clip']:
            for param in self.clip_model.parameters():
                param.requires_grad = False
            print("CLIP weights frozen")
        else:
            # Freeze all except last few layers
            # [TRY NEXT] - Experiment with different unfreezing strategies
            for name, param in self.clip_model.named_parameters():
                if 'ln_final' not in name and 'text_projection' not in name and 'visual.ln_post' not in name:
                    param.requires_grad = False
            print(f"CLIP partially frozen - finetuning final layers only")
    
    # def encode_text(self, text_tokens):
    #     """Encode text using CLIP"""
    #     with torch.cuda.amp.autocast():
    #         text_features = self.clip_model.encode_text(text_tokens)
    #         text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    #     return text_features
    
    # def encode_image(self, images):
    #     """Encode images using CLIP"""
    #     with torch.cuda.amp.autocast():
    #         image_features = self.clip_model.encode_image(images)
    #         image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    #     return image_features
    def encode_text(self, text_tokens):
        """Encode text using CLIP - NO autocast for text"""
        text_features = self.clip_model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features

    def encode_image(self, images):
        """Encode images using CLIP - autocast only for vision"""
        with torch.cuda.amp.autocast():
            image_features = self.clip_model.encode_image(images)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features
    
    def forward(self, text_tokens, images):
        """Extract both text and image features"""
        text_features = self.encode_text(text_tokens)
        image_features = self.encode_image(images)
        return text_features, image_features
    
    # [TRY NEXT] - Add zero-shot classification for categories
    def get_category_probs(self, text_tokens, categories):
        """Get zero-shot classification probabilities for categories"""
        text_features = self.encode_text(text_tokens)
        
        # Encode category names
        category_tokens = clip.tokenize(categories).to(self.device)
        category_features = self.encode_text(category_tokens)
        
        # Compute similarities
        similarities = (100.0 * text_features @ category_features.T).softmax(dim=-1)
        return similarities


In [9]:
# # Cell 6: Cross-Attention Model Architecture
# class MultiHeadCrossAttention(nn.Module):
#     """Cross-attention mechanism for multimodal fusion"""
    
#     def __init__(self, d_model, n_heads=8, dropout=0.1):
#         super().__init__()
#         assert d_model % n_heads == 0
        
#         self.d_model = d_model
#         self.n_heads = n_heads
#         self.d_k = d_model // n_heads
        
#         self.w_q = nn.Linear(d_model, d_model)
#         self.w_k = nn.Linear(d_model, d_model)
#         self.w_v = nn.Linear(d_model, d_model)
#         self.w_o = nn.Linear(d_model, d_model)
        
#         self.dropout = nn.Dropout(dropout)
#         self.layer_norm = nn.LayerNorm(d_model)
        
#         # Learnable temperature for attention scaling
#         self.temperature = nn.Parameter(torch.ones(1))
    
#     def forward(self, query, key, value, mask=None):
#         batch_size = query.size(0)
#         seq_len_q = query.size(1) if len(query.shape) > 2 else 1
#         seq_len_k = key.size(1) if len(key.shape) > 2 else 1
        
#         # Handle both sequential and single vector inputs
#         if len(query.shape) == 2:
#             query = query.unsqueeze(1)
#         if len(key.shape) == 2:
#             key = key.unsqueeze(1)
#         if len(value.shape) == 2:
#             value = value.unsqueeze(1)
        
#         # Linear transformations
#         Q = self.w_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
#         K = self.w_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
#         V = self.w_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
#         # Compute attention scores
#         scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.temperature * np.sqrt(self.d_k))
        
#         if mask is not None:
#             scores = scores.masked_fill(mask == 0, -1e9)
        
#         attn = F.softmax(scores, dim=-1)
#         attn = self.dropout(attn)
        
#         # Apply attention to values
#         context = torch.matmul(attn, V)
#         context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
#         output = self.w_o(context)
#         output = self.dropout(output)
        
#         # Residual connection and layer norm
#         output = self.layer_norm(output + query)
        
#         # Squeeze back if input was 2D
#         if seq_len_q == 1:
#             output = output.squeeze(1)
        
#         return output


# class PricePredictor(nn.Module):
#     """Main model with cross-attention fusion and price prediction"""
    
#     def __init__(self, text_dim=512, vision_dim=512, hidden_dim=512):
#         super().__init__()
        
#         # Feature projections
#         self.text_proj = nn.Linear(text_dim, hidden_dim)
#         self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        
#         # Pack count embedding
#         self.pack_embed = nn.Sequential(
#             nn.Linear(1, 32),
#             nn.ReLU(),
#             nn.Linear(32, 64),
#             nn.LayerNorm(64)
#         )
        
#         # Bidirectional cross-attention
#         self.text_to_vision = MultiHeadCrossAttention(hidden_dim, n_heads=8)
#         self.vision_to_text = MultiHeadCrossAttention(hidden_dim, n_heads=8)
        
#         # Gated fusion
#         self.fusion_gate = nn.Sequential(
#             nn.Linear(hidden_dim * 2 + 64, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(hidden_dim, 2),
#             nn.Softmax(dim=1)
#         )
        
#         # Price prediction head
#         self.price_head = nn.Sequential(
#             nn.Linear(hidden_dim * 2 + 64, hidden_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(hidden_dim, hidden_dim // 2),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(hidden_dim // 2, 1)
#         )
        
#         # Auxiliary task - predict price range (helps with training)
#         self.price_range_head = nn.Linear(hidden_dim // 2, 5)
        
#         # [TRY NEXT] - Add uncertainty estimation
#         # self.uncertainty_head = nn.Linear(hidden_dim // 2, 1)
    
#     def forward(self, text_features, vision_features, pack_count):
#         # Project features
#         text_proj = self.text_proj(text_features)
#         vision_proj = self.vision_proj(vision_features)
        
#         # Cross-attention fusion
#         text_attended = self.text_to_vision(text_proj, vision_proj, vision_proj)
#         vision_attended = self.vision_to_text(vision_proj, text_proj, text_proj)
        
#         # Pack count features
#         pack_features = self.pack_embed(pack_count.unsqueeze(-1))
        
#         # Concatenate all features
#         combined = torch.cat([text_attended, vision_attended, pack_features], dim=-1)
        
#         # Gated fusion
#         gate_weights = self.fusion_gate(combined)
#         fused = gate_weights[:, 0:1] * text_attended + gate_weights[:, 1:2] * vision_attended
        
#         # Final features for prediction
#         final_features = torch.cat([fused, text_attended, vision_attended, pack_features], dim=-1)
        
#         # Price prediction
#         price_pred = self.price_head(final_features)
        
#         return {
#             'price_log': price_pred,
#             'price': torch.expm1(price_pred),  # Convert from log space
#             'attention_weights': gate_weights
#         }
# Fixed Cell 6: Cross-Attention Model Architecture

class MultiHeadCrossAttention(nn.Module):
    """Cross-attention mechanism for multimodal fusion"""
    
    def __init__(self, d_model, n_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)
        
        # Learnable temperature for attention scaling
        self.temperature = nn.Parameter(torch.ones(1))
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Handle both sequential and single vector inputs
        if len(query.shape) == 2:
            query = query.unsqueeze(1)
        if len(key.shape) == 2:
            key = key.unsqueeze(1)
        if len(value.shape) == 2:
            value = value.unsqueeze(1)
        
        # Linear transformations
        Q = self.w_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.temperature * np.sqrt(self.d_k))
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        output = self.w_o(context)
        output = self.dropout(output)
        
        # Residual connection and layer norm
        output = self.layer_norm(output + query)
        
        # Squeeze back if input was 2D
        output = output.squeeze(1)
        
        return output


class PricePredictor(nn.Module):
    """Main model with cross-attention fusion and price prediction"""
    
    def __init__(self, text_dim=512, vision_dim=512, hidden_dim=512):
        super().__init__()
        
        print(f"Initializing PricePredictor with text_dim={text_dim}, vision_dim={vision_dim}, hidden_dim={hidden_dim}")
        
        # Feature projections to common dimension
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.vision_proj = nn.Linear(vision_dim, hidden_dim)
        
        # Pack count embedding (1 -> 64 dim)
        self.pack_embed = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.LayerNorm(64)
        )
        
        # Bidirectional cross-attention
        self.text_to_vision = MultiHeadCrossAttention(hidden_dim, n_heads=8)
        self.vision_to_text = MultiHeadCrossAttention(hidden_dim, n_heads=8)
        
        # Calculate correct input dimension for fusion gate
        # hidden_dim (text_attended) + hidden_dim (vision_attended) + 64 (pack_features)
        fusion_input_dim = hidden_dim * 2 + 64
        
        # Gated fusion
        self.fusion_gate = nn.Sequential(
            nn.Linear(fusion_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2),
            nn.Softmax(dim=1)
        )
        
        # Calculate correct input dimension for price head
        # hidden_dim (fused) + hidden_dim (text_attended) + hidden_dim (vision_attended) + 64 (pack_features)
        price_input_dim = hidden_dim * 3 + 64
        
        # Price prediction head
        self.price_head = nn.Sequential(
            nn.Linear(price_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Auxiliary task - predict price range (helps with training)
        self.price_range_head = nn.Linear(hidden_dim // 2, 5)
        
        print(f"Fusion gate input dim: {fusion_input_dim}")
        print(f"Price head input dim: {price_input_dim}")
        
        # [TRY NEXT] - Add uncertainty estimation
        # self.uncertainty_head = nn.Linear(hidden_dim // 2, 1)
    
    def forward(self, text_features, vision_features, pack_count):
        # Ensure features are 2D tensors
        if len(text_features.shape) == 3:
            text_features = text_features.mean(dim=1)  # Pool if needed
        if len(vision_features.shape) == 3:
            vision_features = vision_features.mean(dim=1)  # Pool if needed
        
        batch_size = text_features.size(0)
        
        # Project features to common dimension
        text_proj = self.text_proj(text_features)  # [batch, hidden_dim]
        vision_proj = self.vision_proj(vision_features)  # [batch, hidden_dim]
        
        # Cross-attention fusion
        text_attended = self.text_to_vision(text_proj, vision_proj, vision_proj)  # [batch, hidden_dim]
        vision_attended = self.vision_to_text(vision_proj, text_proj, text_proj)  # [batch, hidden_dim]
        
        # Pack count features
        pack_features = self.pack_embed(pack_count.unsqueeze(-1))  # [batch, 64]
        
        # Concatenate for gating: text_attended + vision_attended + pack_features
        gate_input = torch.cat([text_attended, vision_attended, pack_features], dim=-1)  # [batch, hidden_dim*2 + 64]
        
        # Gated fusion
        gate_weights = self.fusion_gate(gate_input)  # [batch, 2]
        
        # Apply gating to create fused representation
        fused = gate_weights[:, 0:1] * text_attended + gate_weights[:, 1:2] * vision_attended  # [batch, hidden_dim]
        
        # Final features for prediction: fused + text_attended + vision_attended + pack_features
        final_features = torch.cat([fused, text_attended, vision_attended, pack_features], dim=-1)  # [batch, hidden_dim*3 + 64]
        
        # Price prediction
        price_pred_norm = self.price_head(final_features)  # [batch, 1]
        
        # Denormalize
        price_pred = price_pred_norm * CONFIG['price_log_std'] + CONFIG['price_log_mean']
        
        return {
            'price_log': price_pred,
            'price': torch.expm1(price_pred),  # Convert from log space
            'attention_weights': gate_weights
        }


# Test the model dimensions
def test_model_dimensions():
    """Test if model dimensions are correct"""
    print("Testing model dimensions...")
    
    # Create dummy inputs
    batch_size = 4
    text_dim = 512  # CLIP ViT-B/32 dimension
    vision_dim = 512  # CLIP ViT-B/32 dimension
    
    # Initialize model
    model = PricePredictor(text_dim=text_dim, vision_dim=vision_dim, hidden_dim=512)
    
    # Create dummy inputs
    text_features = torch.randn(batch_size, text_dim)
    vision_features = torch.randn(batch_size, vision_dim)
    pack_counts = torch.randn(batch_size)
    
    # Forward pass
    try:
        output = model(text_features, vision_features, pack_counts)
        print(f"✓ Forward pass successful!")
        print(f"  Price predictions shape: {output['price_log'].shape}")
        print(f"  Attention weights shape: {output['attention_weights'].shape}")
        return True
    except Exception as e:
        print(f"✗ Error in forward pass: {e}")
        return False

# Run dimension test
test_model_dimensions()














# Improved Model Architecture

class ImprovedPricePredictor(nn.Module):
    """Enhanced model with better regularization and feature fusion"""
    
    def __init__(self, text_dim=512, vision_dim=512, hidden_dim=512, dropout=0.3):
        super().__init__()
        
        print(f"Initializing ImprovedPricePredictor")
        
        # Feature projections with LayerNorm
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Enhanced pack count embedding
        self.pack_embed = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.LayerNorm(128)
        )
        
        # Simplified cross-attention (less prone to overfitting)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=4,  # Fewer heads
            dropout=dropout,
            batch_first=True
        )
        
        # Feature fusion with skip connections
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 128, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # Price prediction head with residual
        self.price_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 4, 1)
        )
        
        # Direct path for pack count influence
        self.pack_direct = nn.Linear(128, 1)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Xavier initialization for better gradient flow"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, text_features, vision_features, pack_count):
        # Ensure 2D
        if len(text_features.shape) == 3:
            text_features = text_features.mean(dim=1)
        if len(vision_features.shape) == 3:
            vision_features = vision_features.mean(dim=1)
        
        batch_size = text_features.size(0)
        
        # Project features
        text_proj = self.text_proj(text_features)  # [B, hidden]
        vision_proj = self.vision_proj(vision_features)  # [B, hidden]
        
        # Simple cross-attention (text attends to vision)
        text_attended, _ = self.cross_attention(
            text_proj.unsqueeze(1),  # [B, 1, hidden]
            vision_proj.unsqueeze(1),  # [B, 1, hidden]
            vision_proj.unsqueeze(1)
        )
        text_attended = text_attended.squeeze(1)  # [B, hidden]
        
        # Pack features with more capacity
        pack_features = self.pack_embed(pack_count.unsqueeze(-1))  # [B, 128]
        
        # Combine all features
        combined = torch.cat([text_attended, vision_proj, pack_features], dim=-1)
        
        # Fusion with residual
        fused = self.fusion(combined)
        
        # Price prediction with pack influence
        price_main = self.price_head(fused)
        price_pack_adjustment = self.pack_direct(pack_features)
        
        # Combine predictions (learned weighted sum)
        price_pred = price_main + 0.1 * price_pack_adjustment
        
        return {
            'price_log': price_pred,
            'price': torch.expm1(price_pred * CONFIG.get('price_log_std', 1.0) + CONFIG.get('price_log_mean', 0.0))
        }
        
        #even ssimple for less overfitting ->
        
class CleanPricePredictor(nn.Module):
    def __init__(self, text_dim=512, vision_dim=512, tfidf_dim=50, hidden_dim=256):
        super().__init__()
        
        # Lighter architecture to reduce overfitting
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.5)  # Higher dropout
        )
        
        self.vision_proj = nn.Sequential(
            nn.Linear(vision_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.5)
        )
        
        # TF-IDF projection
        self.tfidf_proj = nn.Sequential(
            nn.Linear(tfidf_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Simple fusion - no complex attention
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
        
        # L2 regularization helper
        self.l2_penalty = 0.0
        
    def forward(self, text_features, vision_features, tfidf_features):
        # Project
        text_h = self.text_proj(text_features)
        vision_h = self.vision_proj(vision_features)
        tfidf_h = self.tfidf_proj(tfidf_features)

        # Concat and predict
        combined = torch.cat([text_h, vision_h, tfidf_h], dim=-1)
        price_log_norm = self.fusion(combined)                  # normalized log μ̂

        # Denormalize for monitoring/preds
        price_log = price_log_norm * CONFIG['price_std'] + CONFIG['price_mean']
        price = torch.expm1(price_log)

        return {
            'price_log_norm': price_log_norm,  # <-- use this in loss
            'price_log': price_log,            # monitoring
            'price': price                     # original scale
        }


Testing model dimensions...
Initializing PricePredictor with text_dim=512, vision_dim=512, hidden_dim=512
Fusion gate input dim: 1088
Price head input dim: 1600
✗ Error in forward pass: 'price_log_std'


In [10]:

# # Cell 7: Loss Functions
# class SMAPELoss(nn.Module):
#     """Symmetric Mean Absolute Percentage Error loss"""
    
#     def __init__(self, epsilon=0.1):
#         super().__init__()
#         self.epsilon = epsilon
    
#     def forward(self, pred, target):
#         # Convert from log space if needed
#         pred_exp = torch.expm1(pred) if pred.max() < 10 else pred
#         target_exp = torch.expm1(target) if target.max() < 10 else target
        
#         numerator = torch.abs(pred_exp - target_exp)
#         denominator = (torch.abs(pred_exp) + torch.abs(target_exp)) / 2.0 + self.epsilon
        
#         smape = numerator / denominator
#         return smape.mean()


# class CombinedLoss(nn.Module):
#     """Combined loss function for robust training"""
    
#     def __init__(self):
#         super().__init__()
#         self.smape = SMAPELoss()
#         self.huber = nn.HuberLoss(delta=1.0)
#         self.mse = nn.MSELoss()
    
#     def forward(self, predictions, targets):
#         price_pred = predictions['price_log'].squeeze()
        
#         # SMAPE loss on actual prices
#         smape_loss = self.smape(price_pred, targets)
        
#         # Huber loss for robustness to outliers
#         huber_loss = self.huber(price_pred, targets)
        
#         # MSE loss in log space for stability
#         mse_loss = self.mse(price_pred, targets)
        
#         # Weighted combination
#         total_loss = (CONFIG['smape_weight'] * smape_loss + 
#                      CONFIG['huber_weight'] * huber_loss + 
#                      CONFIG['log_mse_weight'] * mse_loss)
        
#         return total_loss, {
#             'smape': smape_loss.item(),
#             'huber': huber_loss.item(),
#             'mse': mse_loss.item(),
#             'total': total_loss.item()
#         }

class SMAPEMonitor:
    """SMAPE calculation for monitoring only (not for optimization)"""
    @staticmethod
    def calculate(pred_original_scale, target_original_scale, epsilon=0.1):
        """Calculate SMAPE on original price scale"""
        numerator = torch.abs(pred_original_scale - target_original_scale)
        denominator = (torch.abs(pred_original_scale) + torch.abs(target_original_scale)) / 2.0 + epsilon
        smape = 2.0 * numerator / denominator
        return smape.mean() * 100  # Return as percentage


class CombinedLoss(nn.Module):
    """Combined loss function operating in log space for stability"""
    
    def __init__(self):
        super().__init__()
        self.huber = nn.HuberLoss(delta=1.0)
        self.mse = nn.MSELoss()
        self.smape_monitor = SMAPEMonitor()
    
    def forward(self, predictions, targets_log):
        """
        Args:
            predictions: dict with 'price_log' key
            targets_log: log1p(prices) tensor
        """
        pred_log = predictions['price_log'].squeeze()
        
        # All losses in log space for stability
        huber_loss = self.huber(pred_log, targets_log)
        mse_loss = self.mse(pred_log, targets_log)
        
        # Weighted combination (no SMAPE in optimization)
        total_loss = (CONFIG['huber_weight'] * huber_loss + 
                     CONFIG['log_mse_weight'] * mse_loss)
        
        # SMAPE for monitoring only
        with torch.no_grad():
            pred_original = torch.expm1(pred_log)
            target_original = torch.expm1(targets_log)
            smape_monitor = self.smape_monitor.calculate(pred_original, target_original)
        
        return total_loss, {
            'huber': huber_loss.item(),
            'mse': mse_loss.item(),
            'total': total_loss.item(),
            'smape_monitor': smape_monitor.item()
        }
        
        
        
# Improved Loss Function Cell

class LogNormalNLLLoss(nn.Module):
    """Log-normal negative log-likelihood loss - better for price distributions"""
    
    def __init__(self):
        super().__init__()
        # Learnable uncertainty parameter
        self.log_sigma = nn.Parameter(torch.tensor(0.0))
    
    def forward(self, predictions, targets_normalized):
        """
        Args:
            predictions: dict with 'price_log' (already normalized)
            targets_normalized: normalized log prices
        """
        mu = predictions['price_log'].squeeze()
        
        # Learned uncertainty
        sigma2 = torch.exp(2 * self.log_sigma)
        
        # NLL loss
        nll = 0.5 * torch.mean((targets_normalized - mu)**2 / sigma2 + torch.log(sigma2))
        
        # SMAPE for monitoring (denormalize first)
        with torch.no_grad():
            mu_denorm = mu * CONFIG['price_log_std'] + CONFIG['price_log_mean']
            targets_denorm = targets_normalized * CONFIG['price_log_std'] + CONFIG['price_log_mean']
            
            pred_price = torch.expm1(mu_denorm)
            target_price = torch.expm1(targets_denorm)
            
            smape = 2.0 * torch.abs(pred_price - target_price) / (torch.abs(pred_price) + torch.abs(target_price) + 0.1)
            smape = smape.mean() * 100
        
        return nll, {
            'nll': nll.item(),
            'sigma': torch.exp(self.log_sigma).item(),
            'smape_monitor': smape.item()
        }

# class NormalizedL1Loss(nn.Module):
#     def forward(self, predictions, targets_normalized):
#         mu = predictions['price_log'].squeeze()
#         loss = torch.mean(torch.abs(mu - targets_normalized))
#         with torch.no_grad():
#             mu_den = mu * CONFIG['price_log_std'] + CONFIG['price_log_mean']
#             tgt_den = targets_normalized * CONFIG['price_log_std'] + CONFIG['price_log_mean']
#             smape = (2.0 * torch.abs(torch.expm1(mu_den) - torch.expm1(tgt_den)) /
#                      (torch.abs(torch.expm1(mu_den)) + torch.abs(torch.expm1(tgt_den)) + 0.1)).mean() * 100
#         return loss, {'mae_norm': loss.item(), 'smape_monitor': smape.item()}
class NormalizedL1Loss(nn.Module):
    def forward(self, predictions, targets_normalized):
        mu_norm = predictions['price_log_norm'].squeeze()
        loss = torch.mean(torch.abs(mu_norm - targets_normalized))

        # Monitoring on original scale
        with torch.no_grad():
            mu_den = mu_norm * CONFIG['price_std'] + CONFIG['price_mean']
            tgt_den = targets_normalized * CONFIG['price_std'] + CONFIG['price_mean']
            smape = (2.0 * torch.abs(torch.expm1(mu_den) - torch.expm1(tgt_den)) /
                     (torch.abs(torch.expm1(mu_den)) + torch.abs(torch.expm1(tgt_den)) + 0.1)).mean() * 100
        return loss, {'mae_norm': loss.item(), 'smape_monitor': smape.item()}




class RobustRegressionLoss(nn.Module):
    """Alternative: Robust loss less sensitive to outliers"""
    
    def __init__(self):
        super().__init__()
        self.huber = nn.SmoothL1Loss(beta=0.5)  # More robust than MSE
        
    def forward(self, predictions, targets_normalized):
        mu = predictions['price_log'].squeeze()
        
        # Main loss in normalized space
        loss = self.huber(mu, targets_normalized)
        
        # Add small L2 regularization on extreme predictions
        extreme_penalty = 0.01 * torch.mean(torch.relu(torch.abs(mu) - 3.0))
        
        total_loss = loss + extreme_penalty
        
        # SMAPE monitoring
        with torch.no_grad():
            mu_denorm = mu * CONFIG['price_log_std'] + CONFIG['price_log_mean']
            targets_denorm = targets_normalized * CONFIG['price_log_std'] + CONFIG['price_log_mean']
            
            pred_price = torch.expm1(mu_denorm)
            target_price = torch.expm1(targets_denorm)
            
            smape = 2.0 * torch.abs(pred_price - target_price) / (torch.abs(pred_price) + torch.abs(target_price) + 0.1)
            smape = smape.mean() * 100
        
        return total_loss, {
            'loss': loss.item(),
            'penalty': extreme_penalty.item(),
            'smape_monitor': smape.item()
        }

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


# Cell 8: Data Preparation and Split

def add_tfidf_features(train_df, val_df, test_df, n_components=50):
    vec = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.9)
    X_tr = vec.fit_transform(train_df['catalog_content'].fillna(''))
    X_va = vec.transform(val_df['catalog_content'].fillna(''))
    X_te = vec.transform(test_df['catalog_content'].fillna(''))

    svd = TruncatedSVD(n_components=n_components, random_state=CONFIG['seed'])
    Z_tr = svd.fit_transform(X_tr)
    Z_va = svd.transform(X_va)
    Z_te = svd.transform(X_te)
    return Z_tr.astype('float32'), Z_va.astype('float32'), Z_te.astype('float32')

def prepare_stratified_split(df, val_split=0.2, n_bins=10):
    """Create stratified train/val split based on price ranges"""
    
    # Create price bins for stratification
    df['price_bin'] = pd.qcut(df['price'], q=n_bins, labels=False, duplicates='drop')
    
    # Stratified split
    train_df, val_df = train_test_split(
        df, 
        test_size=val_split,
        stratify=df['price_bin'],
        random_state=CONFIG['seed']
    )
    
    # Drop the temporary bin column
    train_df = train_df.drop('price_bin', axis=1)
    val_df = val_df.drop('price_bin', axis=1)
    
    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Train price range: ${train_df['price'].min():.2f} - ${train_df['price'].max():.2f}")
    print(f"Val price range: ${val_df['price'].min():.2f} - ${val_df['price'].max():.2f}")
    
    return train_df, val_df

# Prepare data splits
train_df_full = pd.read_csv(CONFIG['train_csv'])
train_df, val_df = prepare_stratified_split(
    train_df_full, 
    val_split=CONFIG['val_split'],
    n_bins=CONFIG['stratify_bins']
)


train_prices_log = np.log1p(train_df_full['price'].values)
# CONFIG['price_log_mean'] = float(np.mean(train_prices_log))
# CONFIG['price_log_std']  = float(np.std(train_prices_log) + 1e-8)
CONFIG['price_mean'] = float(train_prices_log.mean())
CONFIG['price_std'] = float(train_prices_log.std())





# If you have test.csv, use it; otherwise use val_df as a placeholder (you can redo for test later)
test_df = pd.read_csv(CONFIG['test_csv']) if os.path.exists(CONFIG['test_csv']) else val_df.copy()

# Build TF-IDF (50D) on train, transform val & test
train_tfidf, val_tfidf, test_tfidf = add_tfidf_features(train_df, val_df, test_df, n_components=50)

# Save TF-IDF features parallel to the CLIP caches (one npz per split)
os.makedirs('outputs', exist_ok=True)
np.savez_compressed('outputs/train_tfidf.npz', tfidf=train_tfidf, ids=train_df['sample_id'].values)
np.savez_compressed('outputs/val_tfidf.npz',   tfidf=val_tfidf,   ids=val_df['sample_id'].values)

# Save test tfidf now if you already have test_df ready (or do this later before inference)
if os.path.exists(CONFIG['test_csv']):
    np.savez_compressed('outputs/test_tfidf.npz',  tfidf=test_tfidf,  ids=test_df['sample_id'].values)






# --- PACK COUNT NORMALIZATION STATS ---
# Compute pack count from catalog text, log-transform, then get mean/std


# Initialize CLIP model and preprocessor
clip_model, clip_preprocess = clip.load(CONFIG['clip_model'], device=CONFIG['device'])
clip_extractor = CLIPFeatureExtractor(CONFIG['clip_model'], CONFIG['device'])

# Create datasets
train_dataset = ProductPriceDataset(
    train_df, 
    CONFIG['image_dir'],
    clip_model,
    clip_preprocess,
    text_preprocessor,
    is_train=True
)

val_dataset = ProductPriceDataset(
    val_df,
    CONFIG['image_dir'], 
    clip_model,
    clip_preprocess,
    text_preprocessor,
    is_train=False
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'] * 2,  # Can use larger batch for validation
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")



Training samples: 60000
Validation samples: 15000
Train price range: $0.13 - $2796.00
Val price range: $0.36 - $1280.00
CLIP text dimension: 512
CLIP vision dimension: 512
CLIP weights frozen
Train batches: 938
Validation batches: 118


In [12]:
# Cell 8.5: Cache CLIP Embeddings for Massive Speedup

import os
import numpy as np
from tqdm import tqdm
def cache_clip_embeddings(df, split_name, clip_extractor, image_dir, text_preprocessor, 
                          batch_size=128, force_recache=False):
    os.makedirs('outputs', exist_ok=True)
    cache_path = f'outputs/{split_name}_embeddings.npz'
    if os.path.exists(cache_path) and not force_recache:
        print(f"Loading cached embeddings from {cache_path}")
        return cache_path

    dataset = ProductPriceDataset(
        df, image_dir, clip_model, clip_preprocess,
        text_preprocessor, is_train=False
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=CONFIG['num_workers'], pin_memory=True)

    all_text_features, all_vision_features, all_prices, all_ids = [], [], [], []

    clip_extractor.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Caching {split_name}"):
            text_tokens = batch['text_tokens'].to(CONFIG['device'])
            images = batch['image'].to(CONFIG['device'])
            tfeat, vfeat = clip_extractor(text_tokens, images)
            all_text_features.append(tfeat.cpu().numpy())
            all_vision_features.append(vfeat.cpu().numpy())
            # log1p price for train/val only
            if 'price' in batch:
                all_prices.append(batch['price'].numpy())
            all_ids.append(batch['sample_id'].numpy())

    save_kwargs = dict(
        text_features=np.concatenate(all_text_features),
        vision_features=np.concatenate(all_vision_features),
        sample_ids=np.concatenate(all_ids),
    )
    if len(all_prices) > 0:
        save_kwargs['prices'] = np.concatenate(all_prices)

    np.savez_compressed(cache_path, **save_kwargs)
    print(f"Saved embeddings to {cache_path}")
    return cache_path


# class CachedEmbeddingDataset(Dataset):
#     """Dataset that loads pre-computed embeddings from disk"""
    
#     def __init__(self, cache_path, is_train=True):
#         data = np.load(cache_path)
#         self.text_features = torch.from_numpy(data['text_features']).float()
#         self.vision_features = torch.from_numpy(data['vision_features']).float()
        
#         # self.pack_counts = torch.from_numpy(data['pack_counts']).float()
#         self.pack_counts = torch.from_numpy(
#             (np.log1p(data['pack_counts']) - CONFIG['pack_mean']) / CONFIG['pack_std']
#         ).float()

        
        
#         self.prices = torch.from_numpy(data['prices']).float()
#         self.sample_ids = data['sample_ids']
#         self.is_train = is_train
#         self.prices = (self.prices - CONFIG['price_log_mean']) / CONFIG['price_log_std']

        
#         print(f"Loaded {len(self.text_features)} cached embeddings")
    
#     def __len__(self):
#         return len(self.text_features)
    
#     def __getitem__(self, idx):
#         # Apply MixUp/CutMix here on embeddings if needed (much faster than on images)
#         return {
#             'text_features': self.text_features[idx],
#             'vision_features': self.vision_features[idx],
#             'pack_count': self.pack_counts[idx],
#             'price': self.prices[idx],
#             'sample_id': self.sample_ids[idx]
#         }

class CachedEmbeddingDataset(Dataset):
    def __init__(self, clip_npz_path, tfidf_npz_path, is_train=True):
        E = np.load(clip_npz_path)
        T = np.load(tfidf_npz_path)

        # core arrays
        text = torch.from_numpy(E['text_features']).float()
        vis  = torch.from_numpy(E['vision_features']).float()
        ids  = E['sample_ids']

        # align TF-IDF by sample_id
        tfidf = T['tfidf']                  # [N_tfidf, 50]
        tfidf_ids = T['ids']                # [N_tfidf]
        # build index for tfidf ids
        idx_map = {int(k): i for i, k in enumerate(tfidf_ids)}
        tfidf_aligned = np.zeros((len(ids), tfidf.shape[1]), dtype=np.float32)
        miss = 0
        for i, sid in enumerate(ids):
            j = idx_map.get(int(sid), None)
            if j is None:
                miss += 1
                continue
            tfidf_aligned[i] = tfidf[j]
        if miss:
            print(f"[TFIDF] missing {miss} rows; filled with zeros")

        self.text_features   = text
        self.vision_features = vis
        self.tfidf_features  = torch.from_numpy(tfidf_aligned).float()
        self.sample_ids      = ids
        self.is_train        = is_train

        # targets (normalized log-price) only for train/val
        if 'prices' in E.files:
            log_prices = E['prices']  # log1p from dataset
            y_norm = (log_prices - CONFIG['price_mean']) / CONFIG['price_std']
            self.prices = torch.from_numpy(y_norm).float()
        else:
            self.prices = None

        print(f"Loaded {len(self.text_features)} cached embeddings with TF-IDF {self.tfidf_features.shape[1]} dims")

    def __len__(self): return len(self.text_features)

    def __getitem__(self, idx):
        item = {
            'text_features': self.text_features[idx],
            'vision_features': self.vision_features[idx],
            'tfidf_features': self.tfidf_features[idx],
            'sample_id': self.sample_ids[idx]
        }
        if self.prices is not None:
            item['price'] = self.prices[idx]
        return item


# Cache embeddings for train and validation
print("\n" + "="*50)
print("CACHING CLIP EMBEDDINGS")
print("="*50)





train_cache = cache_clip_embeddings(
    train_df, 'train', clip_extractor, CONFIG['image_dir'], 
    text_preprocessor, batch_size=128
)

val_cache = cache_clip_embeddings(
    val_df, 'val', clip_extractor, CONFIG['image_dir'],
    text_preprocessor, batch_size=128
)

# # Create cached datasets
# cached_train_dataset = CachedEmbeddingDataset(train_cache, is_train=True)
# cached_val_dataset = CachedEmbeddingDataset(val_cache, is_train=False)


cached_train_dataset = CachedEmbeddingDataset(
    clip_npz_path=train_cache,
    tfidf_npz_path='outputs/train_tfidf.npz',
    is_train=True
)
cached_val_dataset = CachedEmbeddingDataset(
    clip_npz_path=val_cache,
    tfidf_npz_path='outputs/val_tfidf.npz',
    is_train=False
)


# New data loaders with cached embeddings
fast_train_loader = DataLoader(
    cached_train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,  # Less workers needed for cached data
    pin_memory=True,
    persistent_workers=True
)

fast_val_loader = DataLoader(
    cached_val_dataset,
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print(f"\nFast train batches: {len(fast_train_loader)}")
print(f"Fast val batches: {len(fast_val_loader)}")


CACHING CLIP EMBEDDINGS


Caching train: 100%|██████████| 469/469 [10:04<00:00,  1.29s/it]


Saved embeddings to outputs/train_embeddings.npz


Caching val: 100%|██████████| 118/118 [02:02<00:00,  1.04s/it]


Saved embeddings to outputs/val_embeddings.npz
Loaded 60000 cached embeddings with TF-IDF 50 dims
Loaded 15000 cached embeddings with TF-IDF 50 dims

Fast train batches: 938
Fast val batches: 118


In [13]:


torch.backends.cudnn.benchmark = True

# Update CONFIG:
CONFIG['persistent_workers'] = True
CONFIG['prefetch_factor'] = 4

CONFIG['mixup_alpha'] = 0.0  # Disable MixUp
CONFIG['cutmix_alpha'] = 0.0  # Disable CutMix

# # Cell 9: Training Functions with MixUp and CutMix

# class Trainer:
#     """Training loop with advanced augmentations and monitoring"""
    
#     def __init__(self, model, clip_extractor, train_loader, val_loader, config):
#         self.model = model
#         self.clip_extractor = clip_extractor
#         self.train_loader = train_loader
#         self.val_loader = val_loader
#         self.config = config
#         self.device = config['device']
        
#         # Move models to device
#         self.model.to(self.device)
#         self.clip_extractor.to(self.device)
        
#         # Initialize optimizer with different learning rates
#         clip_params = list(self.clip_extractor.parameters())
#         model_params = list(self.model.parameters())
        
#         self.optimizer = torch.optim.AdamW([
#             {'params': clip_params, 'lr': config['learning_rate'] * 0.1},  # Lower LR for CLIP
#             {'params': model_params, 'lr': config['learning_rate']}
#         ], weight_decay=0.01)
        
#         # Learning rate scheduler - Cosine Annealing with Warm Restarts
#         self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#             self.optimizer, 
#             T_0=10,  # Restart every 10 epochs
#             T_mult=2,  # Double the restart interval each time
#             eta_min=1e-6
#         )
        
#         # Loss function
#         self.criterion = CombinedLoss()
        
#         # Mixed precision training
#         self.scaler = torch.cuda.amp.GradScaler()
        
#         # Training history
#         self.history = {
#             'train_loss': [],
#             'val_loss': [],
#             'val_smape': [],
#             'learning_rates': []
#         }
        
#         # Best model tracking
#         self.best_val_loss = float('inf')
#         self.patience_counter = 0
    
#     def mixup_data(self, images, text_tokens, pack_counts, prices, alpha=0.2):
#         """Apply MixUp augmentation"""
#         if alpha > 0:
#             lam = np.random.beta(alpha, alpha)
#         else:
#             lam = 1
        
#         batch_size = images.size(0)
#         index = torch.randperm(batch_size).to(self.device)
        
#         mixed_images = lam * images + (1 - lam) * images[index]
#         # Text tokens stay discrete - we'll mix features instead
#         mixed_prices = lam * prices + (1 - lam) * prices[index]
        
#         return mixed_images, text_tokens, pack_counts, mixed_prices, lam, index
    
#     def train_epoch(self, epoch):
#         """Train for one epoch"""
#         self.model.train()
#         self.clip_extractor.train()
        
#         total_loss = 0
#         loss_components = {'smape': 0, 'huber': 0, 'mse': 0}
        
#         progress_bar = tqdm(self.train_loader, desc=f'Epoch {epoch+1} Training')
        
#         for batch_idx, batch in enumerate(progress_bar):
#             # Move batch to device
#             text_tokens = batch['text_tokens'].to(self.device)
#             images = batch['image'].to(self.device)
#             pack_counts = batch['pack_count'].to(self.device)
#             prices = batch['price'].to(self.device)
            
#             # Apply MixUp with probability
#             if self.config['use_heavy_augmentations'] and np.random.random() < 0.5:
#                 images, text_tokens, pack_counts, prices, lam, index = self.mixup_data(
#                     images, text_tokens, pack_counts, prices, 
#                     alpha=self.config['mixup_alpha']
#                 )
            
#             self.optimizer.zero_grad()
            
#             # Forward pass with mixed precision
#             with torch.cuda.amp.autocast():
#                 # Extract CLIP features
#                 text_features, vision_features = self.clip_extractor(text_tokens, images)
                
#                 # Get predictions
#                 predictions = self.model(text_features, vision_features, pack_counts)
                
#                 # Calculate loss
#                 loss, loss_dict = self.criterion(predictions, prices)
            
#             # Backward pass
#             self.scaler.scale(loss).backward()
            
#             # Gradient clipping
#             self.scaler.unscale_(self.optimizer)
#             torch.nn.utils.clip_grad_norm_(
#                 list(self.clip_extractor.parameters()) + list(self.model.parameters()),
#                 max_norm=1.0
#             )
            
#             self.scaler.step(self.optimizer)
#             self.scaler.update()
            
#             # Update statistics
#             total_loss += loss.item()
#             for key in loss_components:
#                 loss_components[key] += loss_dict[key]
            
#             # Update progress bar
#             progress_bar.set_postfix({
#                 'loss': f'{loss.item():.4f}',
#                 'smape': f'{loss_dict["smape"]:.4f}'
#             })
        
#         # Calculate epoch averages
#         avg_loss = total_loss / len(self.train_loader)
#         for key in loss_components:
#             loss_components[key] /= len(self.train_loader)
        
#         return avg_loss, loss_components
    
#     def validate(self):
#         """Validate the model"""
#         self.model.eval()
#         self.clip_extractor.eval()
        
#         total_loss = 0
#         all_predictions = []
#         all_targets = []
        
#         with torch.no_grad():
#             for batch in tqdm(self.val_loader, desc='Validation'):
#                 # Move batch to device
#                 text_tokens = batch['text_tokens'].to(self.device)
#                 images = batch['image'].to(self.device)
#                 pack_counts = batch['pack_count'].to(self.device)
#                 prices = batch['price'].to(self.device)
#                 prices_original = batch['price_original'].to(self.device)
                
#                 with torch.cuda.amp.autocast():
#                     # Extract features
#                     text_features, vision_features = self.clip_extractor(text_tokens, images)
                    
#                     # Get predictions
#                     predictions = self.model(text_features, vision_features, pack_counts)
                    
#                     # Calculate loss
#                     loss, _ = self.criterion(predictions, prices)
                
#                 total_loss += loss.item()
                
#                 # Store predictions for SMAPE calculation
#                 pred_prices = predictions['price'].squeeze().cpu().numpy()
#                 all_predictions.extend(pred_prices)
#                 all_targets.extend(prices_original.cpu().numpy())
        
#         # Calculate metrics
#         avg_loss = total_loss / len(self.val_loader)
        
#         # Calculate SMAPE on original scale
#         all_predictions = np.array(all_predictions)
#         all_targets = np.array(all_targets)
        
#         smape = np.mean(
#             2.0 * np.abs(all_predictions - all_targets) / 
#             (np.abs(all_predictions) + np.abs(all_targets) + 0.1)
#         ) * 100
        
#         return avg_loss, smape
    
#     def train(self, epochs):
#         """Full training loop"""
#         print(f"\nStarting training for {epochs} epochs...")
#         print(f"Early stopping patience: {self.config['early_stopping_patience']}")
        
#         for epoch in range(epochs):
#             # Training
#             train_loss, train_components = self.train_epoch(epoch)
            
#             # Validation
#             val_loss, val_smape = self.validate()
            
#             # Update learning rate
#             self.scheduler.step()
#             current_lr = self.optimizer.param_groups[0]['lr']
            
#             # Update history
#             self.history['train_loss'].append(train_loss)
#             self.history['val_loss'].append(val_loss)
#             self.history['val_smape'].append(val_smape)
#             self.history['learning_rates'].append(current_lr)
            
#             # Print epoch summary
#             print(f"\nEpoch {epoch+1}/{epochs}")
#             print(f"Train Loss: {train_loss:.4f} (SMAPE: {train_components['smape']:.4f}, "
#                   f"Huber: {train_components['huber']:.4f}, MSE: {train_components['mse']:.4f})")
#             print(f"Val Loss: {val_loss:.4f}, Val SMAPE: {val_smape:.2f}%")
#             print(f"Learning Rate: {current_lr:.6f}")
            
#             # Check for improvement
#             if val_loss < self.best_val_loss:
#                 self.best_val_loss = val_loss
#                 self.patience_counter = 0
                
#                 # Save best model
#                 torch.save({
#                     'epoch': epoch,
#                     'model_state_dict': self.model.state_dict(),
#                     'clip_extractor_state_dict': self.clip_extractor.state_dict(),
#                     'optimizer_state_dict': self.optimizer.state_dict(),
#                     'val_loss': val_loss,
#                     'val_smape': val_smape,
#                     'config': self.config
#                 }, 'best_model.pth')
                
#                 print(f"✓ New best model saved (SMAPE: {val_smape:.2f}%)")
#             else:
#                 self.patience_counter += 1
#                 print(f"No improvement for {self.patience_counter} epochs")
            
#             # Early stopping
#             if self.patience_counter >= self.config['early_stopping_patience']:
#                 print(f"\nEarly stopping triggered after {epoch+1} epochs")
#                 break
        
#         print("\nTraining completed!")
#         print(f"Best validation SMAPE: {min(self.history['val_smape']):.2f}%")
        
#         return self.history

# Cell 9 Update: Enhanced Trainer with Resume

class TrainerWithResume:
    """Training loop with checkpoint resume capability"""
    
    def __init__(self, model, clip_extractor, train_loader, val_loader, config):
        self.model = model
        self.clip_extractor = clip_extractor
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = config['device']
        
        # Move models to device
        self.model.to(self.device)
        self.clip_extractor.to(self.device)
        
        # Set CLIP to eval if frozen
        if config['freeze_clip']:
            self.clip_extractor.eval()
            for p in self.clip_extractor.parameters():
                p.requires_grad = False
        
        # Initialize optimizer
        clip_params = [p for p in self.clip_extractor.parameters() if p.requires_grad]
        model_params = list(self.model.parameters())
        
        # self.optimizer = torch.optim.AdamW([
        #     {'params': clip_params, 'lr': config['learning_rate'] * 0.1} if clip_params else None,
        #     {'params': model_params, 'lr': config['learning_rate']}
        # ], weight_decay=0.01)
        
        
        param_groups = []
        if clip_params:  # only add if there are trainable CLIP params
            param_groups.append({'params': clip_params, 'lr': config['learning_rate'] * 0.1})
        param_groups.append({'params': model_params, 'lr': config['learning_rate']})
        self.optimizer = torch.optim.AdamW(param_groups, weight_decay=0.02)
        
        # Remove None groups
        self.optimizer.param_groups = [g for g in self.optimizer.param_groups if g is not None]
        
        # Learning rate scheduler
        # self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        #     self.optimizer, T_0=10, T_mult=2, eta_min=1e-6
        # )
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        self.optimizer,
        mode='min',
        factor=0.7,
        patience=8,
        min_lr=5e-6,
        verbose=True
    )


        
        # Loss function
        self.criterion = CombinedLoss()
        
        # Mixed precision training
        self.scaler = torch.cuda.amp.GradScaler()
        
        # Training history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_smape': [],
            'learning_rates': []
        }
        
        # Best model tracking
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.start_epoch = 0
    
    def save_checkpoint(self, path, epoch, val_loss, val_smape, is_best=False):
        """Save training checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'clip_extractor_state_dict': self.clip_extractor.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'scaler_state_dict': self.scaler.state_dict(),
            'val_loss': val_loss,
            'val_smape': val_smape,
            'config': self.config,
            'patience_counter': self.patience_counter,
            'best_val_loss': self.best_val_loss,
            'history': self.history
        }
        torch.save(checkpoint, path)
        if is_best:
            print(f"✓ Saved best model to {path} (SMAPE: {val_smape:.2f}%)")
    
    def load_checkpoint(self, path):
        """Load training checkpoint"""
        if not os.path.exists(path):
            print(f"No checkpoint found at {path}")
            return 0
        
        print(f"Loading checkpoint from {path}")
        checkpoint = torch.load(path, map_location=self.device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.clip_extractor.load_state_dict(checkpoint['clip_extractor_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.scaler.load_state_dict(checkpoint['scaler_state_dict'])
        
        self.patience_counter = checkpoint.get('patience_counter', 0)
        self.best_val_loss = checkpoint.get('best_val_loss', float('inf'))
        self.history = checkpoint.get('history', self.history)
        
        start_epoch = checkpoint['epoch'] + 1
        print(f"Resumed from epoch {checkpoint['epoch']} (Val SMAPE: {checkpoint['val_smape']:.2f}%)")
        return start_epoch
    
    def train_epoch_cached(self, epoch):
        """Train epoch using cached embeddings (much faster)"""
        self.model.train()
        # CLIP stays in eval mode if frozen
        
        total_loss = 0
        loss_components = {'huber': 0, 'mse': 0, 'smape_monitor': 0}
        
        progress_bar = tqdm(self.train_loader, desc=f'Epoch {epoch+1} Training')
        
        for batch_idx, batch in enumerate(progress_bar):
            # Move batch to device
            text_features = batch['text_features'].to(self.device)
            vision_features = batch['vision_features'].to(self.device)
            # pack_counts = batch['pack_count'].to(self.device)
            prices = batch['price'].to(self.device)
            
            self.optimizer.zero_grad()
            
            # # Forward pass (no CLIP extraction needed!)
            # with torch.cuda.amp.autocast():
            #     predictions = self.model(text_features, vision_features, pack_counts)
            #     loss, loss_dict = self.criterion(predictions, prices)
            tfidf_features = batch['tfidf_features'].to(self.device)
            prices = batch['price'].to(self.device)

            with torch.cuda.amp.autocast():
                predictions = self.model(text_features, vision_features, tfidf_features)
                loss, loss_dict = self.criterion(predictions, prices)
            # Backward pass
            self.scaler.scale(loss).backward()
            
            # Gradient clipping
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            # Update statistics
            total_loss += loss.item()
            for key in loss_components:
                if key in loss_dict:
                    loss_components[key] += loss_dict[key]
            
            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'smape': f"{loss_dict.get('smape_monitor', 0):.1f}%"
            })
        
        # Calculate epoch averages
        avg_loss = total_loss / len(self.train_loader)
        for key in loss_components:
            loss_components[key] /= len(self.train_loader)
        
        return avg_loss, loss_components
    
    def validate(self):
        """Validate the model"""
        self.model.eval()
        
        total_loss = 0
        all_predictions = []
        all_targets = []
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validation'):
                text_features = batch['text_features'].to(self.device)
                vision_features = batch['vision_features'].to(self.device)
                # pack_counts = batch['pack_count'].to(self.device)
                prices = batch['price'].to(self.device)
                tfidf_features = batch['tfidf_features'].to(self.device)
                
                with torch.cuda.amp.autocast():
                    predictions = self.model(text_features, vision_features, tfidf_features)
                    loss, _ = self.criterion(predictions, prices)
                
                total_loss += loss.item()
                
                # Store predictions for SMAPE
                # pred_prices = predictions['price'].squeeze().cpu().numpy()
                # target_prices = torch.expm1(prices).cpu().numpy()
                pred_prices = predictions['price'].squeeze(-1).detach().cpu().numpy()
                # targets are NORMALIZED log-prices; de-normalize to true log, then expm1
                targets_log = prices * CONFIG['price_std'] + CONFIG['price_mean']
                target_prices = torch.expm1(targets_log).cpu().numpy()
                all_predictions.extend(pred_prices.tolist())
                all_targets.extend(target_prices.tolist())
        
        # Calculate metrics
        avg_loss = total_loss / len(self.val_loader)
        
        # Calculate SMAPE
        all_predictions = np.array(all_predictions)
        all_targets = np.array(all_targets)
        smape = np.mean(2.0 * np.abs(all_predictions - all_targets) / 
                        (np.abs(all_predictions) + np.abs(all_targets) + 0.1)) * 100
        
        return avg_loss, smape
    
    def train(self, epochs, start_epoch=0):
        """Full training loop with resume support"""
        print(f"\nStarting training from epoch {start_epoch} to {epochs}")
        print(f"Early stopping patience: {self.config['early_stopping_patience']}")
        
        os.makedirs('outputs', exist_ok=True)
        
        for epoch in range(start_epoch, epochs):
            # Training
            if hasattr(self.train_loader.dataset, 'text_features'):
                # Use cached embeddings training
                train_loss, train_components = self.train_epoch_cached(epoch)
            else:
                # Fallback to regular training (slower)
                print("Warning: Using non-cached training - will be slower")
                train_loss, train_components = self.train_epoch(epoch)
            
            # Validation
            val_loss, val_smape = self.validate()
            
            # Update learning rate
            # self.scheduler.step()
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_smape'].append(val_smape)
            self.history['learning_rates'].append(current_lr)
            
            # Print epoch summary
            print(f"\nEpoch {epoch+1}/{epochs}")
            print(f"Train Loss: {train_loss:.4f} (Huber: {train_components.get('huber', 0):.4f}, "
                  f"MSE: {train_components.get('mse', 0):.4f})")
            print(f"Val Loss: {val_loss:.4f}, Val SMAPE: {val_smape:.2f}%")
            print(f"Learning Rate: {current_lr:.6f}")
            
            # Save last checkpoint
            self.save_checkpoint('outputs/last_try4.pth', epoch, val_loss, val_smape, is_best=False)
            
            # Check for improvement
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                current_date_time = datetime.now().strftime("%Y%m%d_%H%M%S")
                best_model_path = f'outputs/best_model_try4.pth'
                self.save_checkpoint(f'{best_model_path}', epoch, val_loss, val_smape, is_best=True)
            else:
                self.patience_counter += 1
                print(f"No improvement for {self.patience_counter} epochs")
            
            # Early stopping
            if self.patience_counter >= self.config['early_stopping_patience']:
                print(f"\nEarly stopping triggered after {epoch+1} epochs")
                break
        
        print("\nTraining completed!")
        print(f"Best validation SMAPE: {min(self.history['val_smape']):.2f}%")
        
        return self.history


Totally fair question. Here’s what changed from **try3 → try4**, why it was done, and what it means for your code/training loop.

# What we changed (conceptually)

1. **Targets are now normalized (in log space)**

* **Before (try3):** The model predicted `log(price+1)` directly, and losses (Huber/MSE) were computed on that raw log scale.
* **Now (try4):** We still work in log space, but we **z-score** the log prices:

  * Compute stats once from training data:
    `price_log_mean = mean(log1p(price))`, `price_log_std = std(log1p(price))`
  * **DataLoader** (cached version) feeds **normalized log targets**:
    `y_norm = (log1p(price) - mean) / std`
  * **Model output** is interpreted as **μ = normalized log price**.
  * For reporting/predictions/SMAPE we **de-normalize**:
    `log_price = μ * std + mean` → `price = expm1(log_price)`.

**Why:** Zero-mean, unit-variance targets make the optimization numerically stable, gradients better scaled, and loss curvature friendlier. That alone often yields a solid drop in validation error.

2. **Scheduler switch: Cosine restarts → ReduceLROnPlateau**

* **Before:** Cosine restarts bumped LR back up every T₀ epochs, which was derailing convergence once you reached a plateau.
* **Now:** `ReduceLROnPlateau` **only** decays LR when `val_loss` stalls, and never jumps back up. This tends to improve late-stage convergence for regression.

3. **Loss change: distribution-aware (optional)**

* **Before:** Combined Huber + MSE in log space (works, but treats all errors equally).
* **Now:** You tried **LogNormalNLLLoss**: assume log-prices are approx. Gaussian ⇒ prices are log-normal. The model predicts the mean (and a global uncertainty σ). This matches typical price distributions and penalizes relative errors more sensibly.

  * If you prefer a simpler option, **RobustRegressionLoss** (SmoothL1 on normalized log targets + small extreme penalty) is also added.

4. **Model tweaks (ImprovedPricePredictor)**

* **Before:** Cross-attention in both directions + gated fusion + a large final MLP. Good capacity, but a bit heavy and easier to overfit, especially once embeddings are cached.
* **Now:** A leaner stack tailored for cached features:

  * LayerNorm + Dropout after the first linear projections.
  * One **MultiheadAttention** (text attends to vision) instead of two directions.
  * A stronger **pack_count** pathway (128-dim embed) plus a **direct linear adjustment** (small weight) for pack influence.
  * A smaller fusion MLP with LayerNorm and residual feel.

**Why:** You’re not learning CLIP; you’re learning a *regressor on top of fixed embeddings*. The simpler, regularized fusion behaves better and trains faster/stabler with cached features.

5. **Validation metric calculation is now consistent**

* **Before:** SMAPE sometimes mixed scales (log vs original) in a few places.
* **Now:** We **always** compute SMAPE on the **original price scale**:
  model output (μ) → de-normalize to log → `exp(m1)` to price → SMAPE vs true price.

6. **Small but important fixes**

* The `price_pred_norm` naming (avoid undefined var).
* `.numpy()` needs to be **called**; then `.tolist()`.

---

# Why you “can’t just use the original predictor”

You *can* use it, but if you keep the old architecture **without**:

* target normalization,
* plateau scheduler,
* distribution-aware / robust loss,
  you’ll likely hit the same plateau and LR spikes you saw. The original model can still work if you **keep the old layers but adopt**:
* (A) normalized log targets (as we do now),
* (B) `ReduceLROnPlateau`,
* (C) a better loss (LogNormalNLL or RobustRegressionLoss).

If you want the absolute minimal change path:

* Keep your **original PricePredictor**,
* Ensure **targets are normalized** in `CachedEmbeddingDataset`,
* Make the model output interpreted as **normalized log price** and **de-normalize** inside `forward` (or outside before SMAPE),
* Use **ReduceLROnPlateau**,
* Switch loss to **RobustRegressionLoss** (lowest friction).

That gives you most of the gain without the architectural swap.

---

# TL;DR

* We normalized the **log prices** to mean 0 / std 1 → easier optimization.
* We changed LR schedule to **ReduceLROnPlateau** → no disruptive restarts.
* We moved to a **distribution-aware/robust loss** → better fit for prices.
* We simplified & regularized the **fusion head** → trains cleaner with cached CLIP features.
* We fixed a couple of **bugs** and made SMAPE calc consistent on original scale.

These changes target the exact issues you were seeing: unstable LR after cosine restarts, hard-to-optimize unnormalized targets, and a slightly over-eager fusion block for frozen embeddings.


In [16]:

# Cell 10: Initialize and Train Model
# Initialize model
# model = PricePredictor(
#     text_dim=clip_extractor.text_dim,
#     vision_dim=clip_extractor.vision_dim,
#     hidden_dim=512
# )


# model = ImprovedPricePredictor(
#     text_dim=clip_extractor.text_dim,
#     vision_dim=clip_extractor.vision_dim,
#     hidden_dim=512,
#     dropout=0.3
# )
model = CleanPricePredictor(
    text_dim=clip_extractor.text_dim,
    vision_dim=clip_extractor.vision_dim,
    tfidf_dim=50,
    hidden_dim=256
)


print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"CLIP parameters: {sum(p.numel() for p in clip_extractor.parameters()):,}")

# Initialize trainer
# trainer = Trainer(model, clip_extractor, train_loader, val_loader, CONFIG)

# # Train model
# history = trainer.train(CONFIG['epochs'])

# Initialize with cached data
trainer = TrainerWithResume(model, clip_extractor, fast_train_loader, fast_val_loader, CONFIG)

# trainer.criterion = LogNormalNLLLoss() 
trainer.criterion = NormalizedL1Loss()


# Try to resume from last checkpoint
# start_epoch = trainer.load_checkpoint('outputs/last.pth')
start_epoch = 0

# Train (will resume from where it left off)


history = trainer.train(CONFIG['epochs'], start_epoch=start_epoch)



Model parameters: 447,681
CLIP parameters: 151,277,313

Starting training from epoch 0 to 500
Early stopping patience: 100


Validation: 100%|██████████| 118/118 [00:00<00:00, 450.67it/s]



Epoch 1/500
Train Loss: 0.6779 (Huber: 0.0000, MSE: 0.0000)
Val Loss: 0.6483, Val SMAPE: 60.12%
Learning Rate: 0.000300
✓ Saved best model to outputs/best_model_try4.pth (SMAPE: 60.12%)


Validation: 100%|██████████| 118/118 [00:00<00:00, 475.29it/s]



Epoch 2/500
Train Loss: 0.6370 (Huber: 0.0000, MSE: 0.0000)
Val Loss: 0.6345, Val SMAPE: 58.87%
Learning Rate: 0.000300
✓ Saved best model to outputs/best_model_try4.pth (SMAPE: 58.87%)


...


Epoch 272/500
Train Loss: 0.4348 (Huber: 0.0000, MSE: 0.0000)
Val Loss: 0.5430, Val SMAPE: 50.79%
Learning Rate: 0.000005
No improvement for 98 epochs


Validation: 100%|██████████| 118/118 [00:00<00:00, 577.90it/s]



Epoch 273/500
Train Loss: 0.4368 (Huber: 0.0000, MSE: 0.0000)
Val Loss: 0.5428, Val SMAPE: 50.78%
Learning Rate: 0.000005
No improvement for 99 epochs


Validation: 100%|██████████| 118/118 [00:00<00:00, 624.06it/s]



Epoch 274/500
Train Loss: 0.4383 (Huber: 0.0000, MSE: 0.0000)
Val Loss: 0.5427, Val SMAPE: 50.77%
Learning Rate: 0.000005
No improvement for 100 epochs

Early stopping triggered after 274 epochs

Training completed!
Best validation SMAPE: 50.72%


In [17]:



# def cache_clip_embeddings_test(df, split_name, clip_extractor, image_dir, text_preprocessor,
#                                batch_size=128, force_recache=False):
#     """
#     Cache CLIP embeddings for a split that may NOT have labels (e.g., test).
#     Always saves: text_features, vision_features, pack_counts, sample_ids.
#     """
#     cache_path = f'outputs/{split_name}_embeddings.npz'
#     os.makedirs('outputs', exist_ok=True)

#     if os.path.exists(cache_path) and not force_recache:
#         print(f"[cache] Loading cached embeddings from {cache_path}")
#         return cache_path

#     print(f"[cache] Creating embedding cache for {split_name} ({len(df)} samples)...")

#     # Dataset without train-time augmentations
#     dataset = ProductPriceDataset(
#         df, image_dir, clip_model, clip_preprocess,
#         text_preprocessor, is_train=False
#     )

#     loader = DataLoader(
#         dataset, batch_size=batch_size, shuffle=False,
#         num_workers=CONFIG['num_workers'], pin_memory=True,
#         persistent_workers=True, prefetch_factor=4
#     )

#     all_text_features, all_vision_features = [], []
#     all_pack_counts, all_sample_ids = [], []

#     clip_extractor.eval()
#     with torch.no_grad():
#         for batch in tqdm(loader, desc=f"Caching {split_name}"):
#             text_tokens = batch['text_tokens'].to(CONFIG['device'])
#             images = batch['image'].to(CONFIG['device'])

#             # CLIP features
#             text_features, vision_features = clip_extractor(text_tokens, images)

#             # Move to CPU and store
#             all_text_features.append(text_features.cpu().numpy())
#             all_vision_features.append(vision_features.cpu().numpy())
#             all_pack_counts.append(batch['pack_count'].numpy())
#             all_sample_ids.append(batch['sample_id'].numpy())



#     np.savez_compressed(
#         cache_path,
#         text_features=np.concatenate(all_text_features),
#         vision_features=np.concatenate(all_vision_features),
#         # pack_counts=np.concatenate(all_pack_counts),
#         sample_ids=np.concatenate(all_sample_ids)
#     )
#     print(f"[cache] Saved test embeddings to {cache_path}")
#     return cache_path


# class CachedTestEmbeddingDataset(Dataset):
#     """Test dataset that reads pre-computed embeddings (no labels needed)."""
#     def __init__(self, cache_path):
#         data = np.load(cache_path)
#         self.text_features = torch.from_numpy(data['text_features']).float()
#         self.vision_features = torch.from_numpy(data['vision_features']).float()
#         # self.pack_counts = torch.from_numpy(
#         #     (np.log1p(data['pack_counts']) - CONFIG['pack_mean']) / (CONFIG['pack_std'])
#         # ).float()
#         self.sample_ids = data['sample_ids']

#         print(f"[cache] Loaded {len(self.text_features)} cached TEST embeddings")

#     def __len__(self):
#         return len(self.text_features)

#     def __getitem__(self, idx):
#         return {
#             'text_features': self.text_features[idx],
#             'vision_features': self.vision_features[idx],
#             # 'pack_count': self.pack_counts[idx],
#             'sample_id': self.sample_ids[idx]
#         }


# def predict_from_cached(model, loader, device):
#     """Run inference from cached embeddings loader and return ids + predictions."""
#     model.eval()
#     all_ids, all_preds = [], []

#     with torch.no_grad():
#         for batch in tqdm(loader, desc='Inference (cached test)'):
#             text_features = batch['text_features'].to(device)
#             vision_features = batch['vision_features'].to(device)
#             # pack_counts = batch['pack_count'].to(device)

#             with torch.cuda.amp.autocast():
#                 outputs = model(text_features, vision_features, pack_counts)
#                 pred_prices = outputs['price'].squeeze(-1).detach().cpu().numpy()

#             all_preds.extend(pred_prices.tolist())
#             all_ids.extend(batch['sample_id'].numpy().tolist())

#     return np.array(all_ids), np.array(all_preds)


# # ---------- Load best model ----------
# best_ckpt_path = 'outputs/best_model_try4.pth'  # keep relative path
# assert os.path.exists(best_ckpt_path), f"Checkpoint not found: {best_ckpt_path}"

# ckpt = torch.load(best_ckpt_path, map_location=CONFIG['device'])
# model.load_state_dict(ckpt['model_state_dict'])
# clip_extractor.load_state_dict(ckpt['clip_extractor_state_dict'])
# print(f"Loaded best model from epoch {ckpt['epoch']} (Val SMAPE: {ckpt['val_smape']:.2f}%)")

# # ---------- Load test data & download images (if needed) ----------
# test_df = pd.read_csv(CONFIG['test_csv'])
# print(f"Test samples: {len(test_df)}")

# print("\nDownloading/ensuring test images...")
# _ = download_images_parallel(test_df, CONFIG['image_dir'])

# # ---------- Cache test embeddings ----------
# test_cache = cache_clip_embeddings_test(
#     test_df, 'test', clip_extractor, CONFIG['image_dir'],
#     text_preprocessor, batch_size=128, force_recache=False
# )

# # ---------- Fast test loader from cache ----------
# cached_test_dataset = CachedTestEmbeddingDataset(test_cache)
# fast_test_loader = DataLoader(
#     cached_test_dataset,
#     batch_size=CONFIG['batch_size'] * 2,
#     shuffle=False,
#     num_workers=2,
#     pin_memory=True,
#     persistent_workers=True
# )

# # ---------- Predict ----------
# test_sample_ids, test_predictions = predict_from_cached(model, fast_test_loader, CONFIG['device'])

# # Optional: quick evaluation if labels exist in test.csv (some comps provide them)
# if 'price' in test_df.columns:
#     # Compute SMAPE on original scale for sanity check
#     # Align order by sample_id
#     gt_map = dict(zip(test_df['sample_id'].values, test_df['price'].values))
#     gt = np.array([gt_map[sid] for sid in test_sample_ids])
#     smape_test = np.mean(2.0 * np.abs(test_predictions - gt) / (np.abs(test_predictions) + np.abs(gt) + 0.1)) * 100
#     print(f"[TEST] SMAPE on provided labels: {smape_test:.2f}%")


# === Cell 11: Test Caching + Inference (TF-IDF aware, no pack) ===
import os
import numpy as np
from tqdm import tqdm
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

def cache_clip_embeddings_test(df, split_name, clip_extractor, image_dir, text_preprocessor,
                               batch_size=128, force_recache=False):
    """
    Cache CLIP embeddings for a split that may NOT have labels (e.g., test).
    Saves: text_features, vision_features, sample_ids
    """
    cache_path = f'outputs/{split_name}_embeddings.npz'
    os.makedirs('outputs', exist_ok=True)

    if os.path.exists(cache_path) and not force_recache:
        print(f"[cache] Loading cached embeddings from {cache_path}")
        return cache_path

    print(f"[cache] Creating embedding cache for {split_name} ({len(df)} samples)...")

    dataset = ProductPriceDataset(
        df, image_dir, clip_model, clip_preprocess,
        text_preprocessor, is_train=False
    )
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False,
        num_workers=CONFIG['num_workers'], pin_memory=True,
        persistent_workers=True
    )

    all_text_features, all_vision_features, all_sample_ids = [], [], []

    clip_extractor.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Caching {split_name}"):
            text_tokens = batch['text_tokens'].to(CONFIG['device'])
            images = batch['image'].to(CONFIG['device'])

            # CLIP features
            text_features, vision_features = clip_extractor(text_tokens, images)

            # Move to CPU and store
            all_text_features.append(text_features.cpu().numpy())
            all_vision_features.append(vision_features.cpu().numpy())
            all_sample_ids.append(batch['sample_id'].numpy())

    np.savez_compressed(
        cache_path,
        text_features=np.concatenate(all_text_features),
        vision_features=np.concatenate(all_vision_features),
        sample_ids=np.concatenate(all_sample_ids)
    )
    print(f"[cache] Saved test embeddings to {cache_path}")
    return cache_path


class CachedTestEmbeddingDataset(Dataset):
    """
    Test dataset that reads pre-computed CLIP embeddings and TF-IDF features.
    """
    def __init__(self, clip_npz_path, tfidf_npz_path):
        E = np.load(clip_npz_path)
        T = np.load(tfidf_npz_path)

        # CLIP features
        self.text_features   = torch.from_numpy(E['text_features']).float()
        self.vision_features = torch.from_numpy(E['vision_features']).float()
        self.sample_ids      = E['sample_ids']

        # Align TF-IDF by sample_id
        tfidf        = T['tfidf']    # [N_tfidf, 50]
        tfidf_ids    = T['ids']      # [N_tfidf]
        idx_map = {int(k): i for i, k in enumerate(tfidf_ids)}
        tfidf_aligned = np.zeros((len(self.sample_ids), tfidf.shape[1]), dtype=np.float32)
        miss = 0
        for i, sid in enumerate(self.sample_ids):
            j = idx_map.get(int(sid), None)
            if j is None:
                miss += 1
                continue
            tfidf_aligned[i] = tfidf[j]
        if miss:
            print(f"[TFIDF][test] missing {miss} rows; filled with zeros")
        self.tfidf_features = torch.from_numpy(tfidf_aligned).float()

        print(f"[cache] Loaded TEST embeddings: {len(self.text_features)} rows; TF-IDF dims={self.tfidf_features.shape[1]}")

    def __len__(self): 
        return len(self.text_features)

    def __getitem__(self, idx):
        return {
            'text_features':   self.text_features[idx],
            'vision_features': self.vision_features[idx],
            'tfidf_features':  self.tfidf_features[idx],
            'sample_id':       self.sample_ids[idx]
        }


def predict_from_cached(model, loader, device):
    """Run inference from cached embeddings loader and return ids + predictions."""
    model.eval()
    all_ids, all_preds = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Inference (cached test)'):
            text_features   = batch['text_features'].to(device)
            vision_features = batch['vision_features'].to(device)
            tfidf_features  = batch['tfidf_features'].to(device)

            with torch.cuda.amp.autocast():
                outputs = model(text_features, vision_features, tfidf_features)
                pred_prices = outputs['price'].squeeze(-1).detach().cpu().numpy()

            all_preds.extend(pred_prices.tolist())
            all_ids.extend(batch['sample_id'].numpy().tolist())

    return np.array(all_ids), np.array(all_preds)


# ---------- Load best model ----------
best_ckpt_path = 'outputs/best_model_try4.pth'
assert os.path.exists(best_ckpt_path), f"Checkpoint not found: {best_ckpt_path}"
ckpt = torch.load(best_ckpt_path, map_location=CONFIG['device'])
model.load_state_dict(ckpt['model_state_dict'])
clip_extractor.load_state_dict(ckpt['clip_extractor_state_dict'])
print(f"Loaded best model from epoch {ckpt['epoch']} (Val SMAPE: {ckpt['val_smape']:.2f}%)")

# ---------- Load test data & download images (if needed) ----------
test_df = pd.read_csv(CONFIG['test_csv'])
print(f"Test samples: {len(test_df)}")
print("\nDownloading/ensuring test images...")
_ = download_images_parallel(test_df, CONFIG['image_dir'])

# ---------- Cache test embeddings ----------
test_cache = cache_clip_embeddings_test(
    test_df, 'test', clip_extractor, CONFIG['image_dir'],
    text_preprocessor, batch_size=128, force_recache=False
)

# ---------- Ensure test TF-IDF exists (created earlier in Cell 8) ----------
test_tfidf_npz = 'outputs/test_tfidf.npz'
if not os.path.exists(test_tfidf_npz):
    # Fallback: recompute TF-IDF for test using train fit if needed
    print("[TFIDF] outputs/test_tfidf.npz missing — recomputing from train fit...")
    _tr, _va, _te = add_tfidf_features(train_df, val_df, test_df, n_components=50)
    np.savez_compressed(test_tfidf_npz, tfidf=_te, ids=test_df['sample_id'].values)

# ---------- Fast test loader from cache ----------
cached_test_dataset = CachedTestEmbeddingDataset(test_cache, test_tfidf_npz)
fast_test_loader = DataLoader(
    cached_test_dataset,
    batch_size=CONFIG['batch_size'] * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

# ---------- Predict ----------
test_sample_ids, test_predictions = predict_from_cached(model, fast_test_loader, CONFIG['device'])

# Optional sanity check if labels exist in test.csv
if 'price' in test_df.columns:
    gt_map = dict(zip(test_df['sample_id'].values, test_df['price'].values))
    gt = np.array([gt_map[sid] for sid in test_sample_ids])
    smape_test = np.mean(2.0 * np.abs(test_predictions - gt) / (np.abs(test_predictions) + np.abs(gt) + 0.1)) * 100
    print(f"[TEST] SMAPE on provided labels: {smape_test:.2f}%")


Loaded best model from epoch 173 (Val SMAPE: 50.72%)
Test samples: 75000

Downloading/ensuring test images...


Failed to download 1 images. See image_download_fail.txt
[cache] Creating embedding cache for test (75000 samples)...


Caching test: 100%|██████████| 586/586 [12:08<00:00,  1.24s/it]


[cache] Saved test embeddings to outputs/test_embeddings.npz
[cache] Loaded TEST embeddings: 75000 rows; TF-IDF dims=50


Inference (cached test): 100%|██████████| 586/586 [00:01<00:00, 486.42it/s]


In [18]:
# === Cell 12: Post-processing + Submission CSV ===
def post_process_predictions(preds, train_df):
    """
    Conservative clipping to training price range (+/- 20%) and rounding to 2 decimals.
    """
    train_min = float(train_df['price'].min())
    train_max = float(train_df['price'].max())
    preds = np.clip(preds, train_min * 0.8, train_max * 1.2)
    return np.round(preds, 2)

final_test_predictions = post_process_predictions(test_predictions, train_df_full)

submission_df = pd.DataFrame({
    'sample_id': test_sample_ids.astype(int),
    'price': final_test_predictions.astype(float)
}).sort_values('sample_id').reset_index(drop=True)

os.makedirs(CONFIG['output_dir'], exist_ok=True)
submission_path = os.path.join(CONFIG['output_dir'], 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f"\n[SUBMIT] Wrote submission file to: {submission_path}")
print(f"[SUBMIT] Shape: {submission_df.shape}")
print(f"[SUBMIT] Price range: ${submission_df['price'].min():.2f} - ${submission_df['price'].max():.2f}")
print("\nSample rows:")
print(submission_df.head(10))



[SUBMIT] Wrote submission file to: outputs/submission.csv
[SUBMIT] Shape: (75000, 2)
[SUBMIT] Price range: $0.10 - $269.85

Sample rows:
   sample_id  price
0          1  26.89
1          3  21.36
2          9  20.93
3         19  17.25
4         20  28.51
5         21  11.37
6         23  14.22
7         25  10.62
8         27  10.83
9         28  13.08
